In [ ]:
# Removendo mensagens de erro:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*is_datetime64tz_dtype is deprecated.*",
    category=DeprecationWarning,
)
# Limpando outputs!
from IPython.display import clear_output

## 1. Extraindo os Dados:

#### Importando bibliotecas essenciais para a Análise de Dados:

In [ ]:
# Importando bibliotecas:
from collections import defaultdict                         # Corrigindo o Headcount e Turnover...
from __future__ import annotations                          # Adia a avaliação das type hints (permite "forward references", ajuda a evitar imports circulares e melhora performance em alguns casos)...
import matplotlib.pyplot as plt                             # Biblioteca para uma visualização mais analítica...
from pathlib import Path                                    # Pegando o endereço do arquivo...
from glob import glob                                       # Busca de arquivos por padrões...
import pandas as pd                                         # Análise e manipulação de dados...
import numpy as np                                          # Tratamento de Dados...
import warnings                                             # Controle de avisos...
import sqlite3                                              # Biblioteca para criar a base de dados em Data Warehouse...
import time                                                 # Biblioteca para corrigir datas e horas...
import os                                                   # Operações com arquivos e sistema...
import re                                                   # Corrigindo o Headcount e Turnover...

pd.set_option('mode.chained_assignment', None)              # Desativa avisos de atribuição encadeada...
warnings.filterwarnings("ignore", category=FutureWarning)   # Oculta avisos de futura depreciação...


#### Consumindo os Dados dos arquivos Parquet:

In [ ]:
# Caminhos das pastas com os arquivos '.parquet':
pasta_ativos = R"..\Base de Dados Original\parquet\Ativos\Ativos.parquet"
pasta_desligados = R"..\Base de Dados Original\parquet\Desligados\Desligados.parquet"
pasta_mensalistas = R"..\Base de Dados Original\parquet\Mensalistas"
pasta_gptw = R"..\Base de Dados Original\parquet\GPTW\GPTW.parquet"
pasta_executivos = R"..\Base de Dados Original\parquet\Executivos"
pasta_horistas = R"..\Base de Dados Original\parquet\Horistas"
pasta_movimentacoes = R"..\Base de Dados Original\parquet\Movimentacoes\Movimentacoes.parquet"

# Lista todos os arquivos '.parquet' em cada pasta:
arquivos_parquet_executivos = glob(os.path.join(pasta_executivos, "*.parquet"))
arquivos_parquet_mensalistas = glob(os.path.join(pasta_mensalistas, "*.parquet"))
arquivos_parquet_horistas = glob(os.path.join(pasta_horistas, "*.parquet"))

# Carrega os arquivos e adiciona a coluna 'Cargo':
tables = [
    pd.read_parquet(caminho).assign(Cargo=cargo)
    for arquivos, cargo in zip([arquivos_parquet_executivos, arquivos_parquet_mensalistas, arquivos_parquet_horistas], ['E', 'M', 'H'])
    for caminho in arquivos
]

# Carrega o arquivo para dentro do Data Frame:
df_desligados = pd.read_parquet(pasta_desligados)
df_ativos = pd.read_parquet(pasta_ativos)
df_gptw = pd.read_parquet(pasta_gptw)
df_movimentacoes = pd.read_parquet(pasta_movimentacoes)

In [ ]:
# === Resumo final ===
total_linhas = sum(len(df) for df in tables)
print("\n📊 RESUMO GERAL")
print(f"Arquivos executivos: {len(arquivos_parquet_executivos)}")
print(f"Arquivos mensalistas: {len(arquivos_parquet_mensalistas)}")
print(f"Arquivos horistas: {len(arquivos_parquet_horistas)}")
print(f"Total de DataFrames lidos: {len(tables)}")
print(f"Total de linhas combinadas: {total_linhas:,}")

## 2. Tratando os Dados:

#### Definição das Funções para Transformação e Tratamento de Dados:
1. **Renomear Colunas:**  
   Função que renomeia as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.
2. **Mover Colunas:**  
   Função que move as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.

In [ ]:
# Função para renomear coluna:
def renomear_coluna(df, col, newname):
    df = df.rename(columns={col: newname})
    return df

# Função para mover coluna
def mover_coluna(df, col, nova_posicao):
    colunas = list(df.columns)
    colunas.insert(nova_posicao, colunas.pop(colunas.index(col)))
    return df[colunas]

#### Aplicando Funções a lista de tabelas:

In [ ]:
# Aplicando a função a todas as colunas de todas as tabelas:
for i in range(len(tables)):
    clear_output(wait=True)
    print(f"Processando tabela {i+1}/{len(tables)}", end="", flush=True)
    tables[i] = renomear_coluna(tables[i], tables[i].columns[0], 'Cadastro')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[1], 'Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[2], 'Descricao Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[3], 'Nome')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[4], 'Filial')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[5], 'Data')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[6], 'Hora')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[7], 'Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[8], 'Dir')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[9], 'Tipo')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[10], 'Uso Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[11], 'Planta')
#'Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Coletor', 'Planta', 'Descricao Coletor', 'Uso Acesso', 'Cargo'...

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_ativos':
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[0], 'Cadastro')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[1], 'Cargo')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[2], 'Nome completo')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[3], 'Heranca')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[4], 'Cargo Especifico')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[5], 'Expatriado')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[6], 'Departamento')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[7], 'Área')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[8], 'Local')

# Aplicando a função de renomear o nome da coluna 'Departamento' da tabela 'df_desligados':
df_desligados = mover_coluna(df_desligados, 'Diretoria RH', 2)
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[0], 'Cadastro')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[2], 'Heranca')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[3], 'Cargo Especifico')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[4], 'Departamento')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[5], 'Área')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[6], 'Local')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_gptw':
df_gptw = renomear_coluna(df_gptw, df_gptw.columns[0], 'Área')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_movimentacoes':
df_movimentacoes = renomear_coluna(df_movimentacoes, df_movimentacoes.columns[0], 'Cadastro')

#### Concatenar todas as tabelas da lista em um único Data Frame:

In [ ]:
# Concatenando todas as tabelas em um único DataFrame:
df_ronda = pd.concat(tables, ignore_index=True)

# Ordenando as colunas na ordem desejada:
df_ronda = df_ronda[['Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo']]
df_movimentacoes = df_movimentacoes[
    ['Cadastro', 'Abreviação UO antiga', 'Desc. UO antiga', 'Desc. Cargo antigo',
    'Abreviação UO', 'Desc. UO', 'Desc. Cargo', 'Tipo da Movimentação', 'Data de alteração']
]
df_ronda = df_ronda.drop(df_ronda.index[-1]) # Remover última linha de total...

#### Diferenciando as pessoas que foram demitidas ou que não serão analisadas e inserindo todos os df_ativos delas:

In [ ]:
# Criando toda a função para encontrar as pessoas desligadas não registradas nas bases 'df_ativos' e 'df_desligados':
TARGET_COLS = [
    "Cadastro", "Cargo",
    "Nome completo", "Heranca", "Cargo Especifico",
    "Departamento", "Área", "Local", "Data_Adm", "Data_Dem"
]

def _to_dt(s, dayfirst=True):
    return pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)

def _coalesce(a, b):
    return a.where(~a.isna(), b)

def _fix_cols_ativos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Garante que df_ativos tenha:
      - colunas únicas (sem duplicadas)
      - 'Cargo' e 'Cargo Especifico' separados (se vierem duplicados como 'Cargo Especifico')
    Regra aplicada:
      - Se houver 2+ 'Cargo Especifico' e não existir 'Cargo', renomeia a 1ª ocorrência para 'Cargo'
      - Se houver só 1 'Cargo Especifico' e não existir 'Cargo', cria 'Cargo' copiando 'Cargo Especifico'
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    cols = pd.Series(df.columns)
    mask_ce = cols.eq("Cargo Especifico")

    if "Cargo" not in cols.values:
        if mask_ce.sum() >= 2:
            idxs = np.flatnonzero(mask_ce.values)
            cols.iloc[idxs[0]] = "Cargo"  # 1ª ocorrência vira Cargo
        elif mask_ce.sum() == 1:
            # cria Cargo copiando Cargo Especifico (melhor do que NA)
            df["Cargo"] = df["Cargo Especifico"]

    df.columns = cols.tolist()

    # remove duplicadas (se ainda restar alguma)
    df = df.loc[:, ~df.columns.duplicated(keep="last")]
    return df

def _dedup_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    return df.loc[:, ~df.columns.duplicated(keep="last")]

def sincronizar_desligamentos(df_movimentacoes, df_ativos, df_desligados, verbose=True):
    mov = _dedup_cols(df_movimentacoes)
    ativos = _fix_cols_ativos(df_ativos)          # <-- AQUI resolve "Cargo Especifico" duplicado
    deslig = _dedup_cols(df_desligados)

    # --- Normaliza Cadastro (Int64 compatível com NA) ---
    for df in (mov, ativos, deslig):
        df["Cadastro"] = pd.to_numeric(df["Cadastro"], errors="coerce").astype("Int64")

    # --- Garante que df_desligados tenha todas as colunas alvo ---
    for c in TARGET_COLS:
        if c not in deslig.columns:
            deslig[c] = pd.NA
    deslig = deslig[TARGET_COLS].copy()

    # --- Parse datas existentes do df_desligados ---
    deslig["Data_Adm_dt"] = _to_dt(deslig["Data_Adm"], dayfirst=True)
    deslig["Data_Dem_dt"] = _to_dt(deslig["Data_Dem"], dayfirst=True)

    # --- Filtra desligamentos nas movimentações ---
    mov["Tipo da Movimentação"] = mov["Tipo da Movimentação"].astype(str).str.strip()
    mov["Data de alteração_dt"] = _to_dt(mov["Data de alteração"], dayfirst=True)

    mov_desl = mov[mov["Tipo da Movimentação"].str.contains("deslig", case=False, na=False)].copy()

    if mov_desl.empty:
        if verbose:
            print("ℹ️ Nenhum 'Desligamento' encontrado em df_movimentacoes.")
        deslig = deslig.drop(columns=["Data_Adm_dt", "Data_Dem_dt"], errors="ignore")
        return ativos, deslig

    cols_mov_util = ["Cadastro", "Data de alteração_dt"]

    if "Cargo" in mov_desl.columns:
        cols_mov_util.append("Cargo")

    for cand in ["Abreviação UO", "Abreviação UO antiga"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break
    for cand in ["Desc. UO", "Desc. UO antiga"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break
    for cand in ["Desc. Cargo", "Desc. Cargo antigo"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break

    mov_last = (
        mov_desl.sort_values("Data de alteração_dt")
               .groupby("Cadastro", as_index=False)
               .tail(1)[cols_mov_util]
               .rename(columns={"Data de alteração_dt": "Data_Dem_dt"})
    )

    incoming = pd.DataFrame({"Cadastro": mov_last["Cadastro"]})
    incoming["Cargo"] = mov_last["Cargo"] if "Cargo" in mov_last.columns else pd.NA

    incoming["Nome completo"] = pd.NA
    incoming["Heranca"] = pd.NA
    incoming["Local"] = pd.NA
    incoming["Data_Adm_dt"] = pd.NaT
    incoming["Data_Dem_dt"] = mov_last["Data_Dem_dt"]

    col_abrev = next((c for c in ["Abreviação UO", "Abreviação UO antiga"] if c in mov_last.columns), None)
    col_uo = next((c for c in ["Desc. UO", "Desc. UO antiga"] if c in mov_last.columns), None)
    col_cargo = next((c for c in ["Desc. Cargo", "Desc. Cargo antigo"] if c in mov_last.columns), None)

    incoming["Área"] = mov_last[col_abrev] if col_abrev else pd.NA
    incoming["Departamento"] = mov_last[col_uo] if col_uo else pd.NA
    incoming["Cargo Especifico"] = mov_last[col_cargo] if col_cargo else pd.NA

    ativos["Data_Admissão_dt"] = _to_dt(ativos.get("Data_Admissão"), dayfirst=True)

    in_ativos = incoming.merge(ativos, on="Cadastro", how="left", suffixes=("", "_a"))

    # (opcional mas recomendado) puxar também o Cargo de ativos, se veio NA em mov
    incoming["Cargo"] = _coalesce(incoming["Cargo"], in_ativos.get("Cargo_a"))

    incoming["Nome completo"] = _coalesce(incoming["Nome completo"], in_ativos.get("Nome completo_a"))
    incoming["Heranca"] = _coalesce(incoming["Heranca"], in_ativos.get("Heranca_a"))
    incoming["Cargo Especifico"] = _coalesce(incoming["Cargo Especifico"], in_ativos.get("Cargo Especifico_a"))
    incoming["Departamento"] = _coalesce(incoming["Departamento"], in_ativos.get("Departamento_a"))
    incoming["Área"] = _coalesce(incoming["Área"], in_ativos.get("Área_a"))
    incoming["Local"] = _coalesce(incoming["Local"], in_ativos.get("Local_a"))
    incoming["Data_Adm_dt"] = _coalesce(incoming["Data_Adm_dt"], in_ativos.get("Data_Admissão_dt"))

    cad_mover = incoming["Cadastro"].dropna().unique()
    ativos_out = ativos[~ativos["Cadastro"].isin(cad_mover)].reset_index(drop=True)

    base = deslig.copy()

    merged = base.merge(
        incoming,
        on="Cadastro",
        how="outer",
        suffixes=("_old", "_new")
    )

    for col in ["Cargo", "Nome completo", "Heranca", "Cargo Especifico", "Departamento", "Área", "Local"]:
        merged[col] = _coalesce(merged.get(f"{col}_old"), merged.get(f"{col}_new"))

    merged["Data_Adm_dt"] = _coalesce(merged.get("Data_Adm_dt_old"), merged.get("Data_Adm_dt_new"))

    old_dem = merged.get("Data_Dem_dt_old")
    new_dem = merged.get("Data_Dem_dt_new")
    merged["Data_Dem_dt"] = old_dem
    merged.loc[merged["Data_Dem_dt"].isna(), "Data_Dem_dt"] = new_dem
    both = old_dem.notna() & new_dem.notna()
    merged.loc[both, "Data_Dem_dt"] = pd.concat([old_dem[both], new_dem[both]], axis=1).max(axis=1)

    merged["Data_Adm"] = pd.to_datetime(merged["Data_Adm_dt"], errors="coerce").dt.strftime("%d.%m.%Y")
    merged["Data_Dem"] = pd.to_datetime(merged["Data_Dem_dt"], errors="coerce").dt.strftime("%d.%m.%Y")

    desligados_out = merged[TARGET_COLS].reset_index(drop=True)

    if verbose:
        print(f"🔎 Desligamentos (movimentações): {len(incoming)}")
        removidos = df_ativos["Cadastro"].isin(cad_mover).sum() if "Cadastro" in df_ativos.columns else 0
        print(f"🧹 Removidos de ativos (se existiam): {removidos}")
        print(f"✅ Ativos: {len(df_ativos)} -> {len(ativos_out)}")
        print(f"✅ Desligados: {len(df_desligados)} -> {len(desligados_out)}")

    ativos_out = ativos_out.drop(columns=["Data_Admissão_dt"], errors="ignore")
    return ativos_out, desligados_out


In [ ]:
# Executando a Função:
df_ativos, df_desligados = sincronizar_desligamentos(
    df_movimentacoes, df_ativos, df_desligados, verbose=True
)


In [ ]:
# Normalização mínima do 'Cadastro' (evita 123, "123", 123.0 divergirem)
def _norm_id(s):
    s = s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    return s.replace({'nan': np.nan})  # volta NaN onde era NaN

for _df in (df_ronda, df_desligados, df_ativos):
    if 'Cadastro' not in _df.columns:
        raise KeyError("Faltou a coluna 'Cadastro' em um dos DataFrames.")
    _df['Cadastro'] = _norm_id(_df['Cadastro'])

# Desligado por Cadastro:
df_ronda['Desligado'] = df_ronda['Cadastro'].isin(df_desligados['Cadastro']).map({True: 'Sim', False: 'Não'})

# Preparar 'df_ativos' com nomes únicos de coluna:
deps = df_ativos[['Cadastro','Nome completo','Área','Departamento','Heranca','Cargo Especifico']].rename(
    columns={
        'Nome completo':'Nome_dep',
        'Área':'Área_dep',
        'Departamento':'Departamento_dep',
        'Heranca':'Heranca_dep',
        'Cargo Especifico':'Cargo Especifico_dep'
    }
)


In [ ]:
# Merge (left) pelo Cadastro:
df_ronda = df_ronda.merge(deps, on='Cadastro', how='left')


In [ ]:
# Dicionários (df_desligados) por Cadastro:
map_df_desligados_nome = dict(zip(df_desligados['Cadastro'], df_desligados['Nome completo']))
map_df_desligados_area = dict(zip(df_desligados['Cadastro'], df_desligados['Área']))
map_df_desligados_dept = dict(zip(df_desligados['Cadastro'], df_desligados['Departamento']))
map_df_desligados_her  = dict(zip(df_desligados['Cadastro'], df_desligados['Heranca']))
map_df_desligados_carg = dict(zip(df_desligados['Cadastro'], df_desligados['Cargo Especifico']))

# Substituir com prioridade: df_desligados > df_ativos > original:

# Nome:
df_ronda['Nome'] = (
    df_ronda['Cadastro'].map(map_df_desligados_nome)   # 1º: df_desligados…
    .fillna(df_ronda['Nome_dep'])                  # 2º: df_ativos…
    .fillna(df_ronda.get('Nome'))                  # 3º: valor original (se existir)…
)

# Área:
df_ronda['Área'] = (
    df_ronda['Cadastro'].map(map_df_desligados_area)
    .fillna(df_ronda['Área_dep'])
)

# Departamento:
df_ronda['Departamento'] = (
    df_ronda['Cadastro'].map(map_df_desligados_dept)
    .fillna(df_ronda['Departamento_dep'])
)

# Heranca:
df_ronda['Heranca'] = (
    df_ronda['Cadastro'].map(map_df_desligados_her)
    .fillna(df_ronda['Heranca_dep'])
)

# Cargo Especifico:
df_ronda['Cargo Especifico'] = (
    df_ronda['Cadastro'].map(map_df_desligados_carg)
    .fillna(df_ronda['Cargo Especifico_dep'])
)

# (Opcional) Manter as colunas *_dep para auditoria; se quiser remover, descomente abaixo:
df_ronda = df_ronda.drop(columns=['Nome_dep','Área_dep','Departamento_dep','Heranca_dep','Cargo Especifico_dep'], errors='ignore')

#### Corrigindo as colunas de Datas:

In [ ]:
# Pareseand por regex:
s = df_ronda["Data"].astype(str).str.strip()

# só datas com barra (ex: 02/09/2026)
mask_slash = s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$")

# tenta BR primeiro (dd/mm/yyyy) sem warning, porque o format é explícito
br = pd.to_datetime(s.where(mask_slash), format="%d/%m/%Y", errors="coerce")

# onde falhou, tenta US (mm/dd/yyyy) também sem warning
us = pd.to_datetime(s.where(mask_slash), format="%m/%d/%Y", errors="coerce")

# para outros formatos (ex: YYYY-MM-DD), tenta o parser padrão
other = pd.to_datetime(s.where(~mask_slash), errors="coerce")

df_ronda["Data"] = other.fillna(br.fillna(us))


In [ ]:
# Hora (minutos -> time):
df_ronda['Hora'] = pd.to_numeric(df_ronda['Hora'], errors='coerce')
df_ronda['Hora'] = pd.to_timedelta(df_ronda['Hora'], unit='m')
df_ronda['Hora'] = (pd.Timestamp('1900-01-01') + df_ronda['Hora']).dt.time

# Data (MANTER como datetime — não usar strftime aqui):
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'], dayfirst=True, errors='coerce')

# Admissão/Demissão (datetime):
df_desligados['Data_Adm'] = pd.to_datetime(df_desligados['Data_Adm'], dayfirst=True, errors='coerce')
df_desligados['Data_Dem'] = pd.to_datetime(df_desligados['Data_Dem'], dayfirst=True, errors='coerce')
df_ativos['Data_Admissão'] = pd.to_datetime(df_ativos['Data_Admissão'], format='%d.%m.%Y', errors='coerce')

def fmt_dt(x):
    return x.strftime('%d/%m/%Y') if pd.notna(x) else 'NaT'

print(f"df_ronda['Data']           -> min={fmt_dt(df_ronda['Data'].min())} | max={fmt_dt(df_ronda['Data'].max())}")
print(f"df_desligados['Data_Adm']  -> min={fmt_dt(df_desligados['Data_Adm'].min())} | max={fmt_dt(df_desligados['Data_Adm'].max())}")
print(f"df_desligados['Data_Dem']  -> min={fmt_dt(df_desligados['Data_Dem'].min())} | max={fmt_dt(df_desligados['Data_Dem'].max())}")
print(f"df_ativos['Data_Admissão'] -> min={fmt_dt(df_ativos['Data_Admissão'].min())} | max={fmt_dt(df_ativos['Data_Admissão'].max())}")

## 3. Carregando os Dados:

### Desenvolvendo toda a Engenharia dos Dados em formato de Data Warehouse:

In [ ]:
# ETL "executável" (sem função principal) para construir o DW SQLite `dw_ronda.sqlite` a partir do df_ronda.
"""
✅ Foco: performance no APP (Dash) via tabelas agregadas MATERIALIZADAS (agg_*).
✅ O app abre o arquivo em READ-ONLY / immutable, sem DDL.

Pré-requisitos no Notebook:
  - df_ronda (obrigatório)
  - df_movimentacoes (obrigatório)
  - df_gptw (obrigatório)
  - df_headcount (opcional)  # se não passar, tenta calcular via df_ativos/df_desligados; senão usa fallback por presença
  - df_ativos (obrigatório; RH)   # snapshot atual (Cadastro, Área, Heranca, Data_Admissão)
  - df_desligados (obrigatório; RH)  # histórico (Cadastro, Área, Heranca, Data_Adm, Data_Dem)

Como usar (no fim do arquivo tem um bloco "EXECUÇÃO"):
  - ajuste SQLITE_PATH e CHUNK_SIZE
  - garanta que df_ronda exista em memória
  - rode o script/célula

Obs: Datas ficam como TEXT ISO (YYYY-MM-DD) no SQLite (ótimo p/ índice). No app, parse com pd.to_datetime.
"""

# =========================
# CONFIG (AJUSTE AQUI)
# =========================
SQLITE_PATH = Path(r"..\project\database\dw_ronda.sqlite")
CHUNK_SIZE = 250_000  # ajuste conforme RAM

# Otimizações (tuning)
FAST_ETL_PRAGMAS = True   # acelera build (recomendado; o DW é reconstruível)
DO_ANALYZE = True        # gera estatísticas p/ planner
DO_OPTIMIZE = True       # PRAGMA optimize (SQLite 3.18+)
DO_VACUUM = False        # compacta arquivo (custo alto no ETL)

# Se True, remove a tabela gigante 'evento_acesso_flat' do DW final (serving-only)
SERVING_ONLY = True
VACUUM_ON_SERVING_ONLY = True  # compacta após dropar (recomendado)

# =========================
# HELPERS
# =========================
def normalize_columns_inplace(df: pd.DataFrame) -> None:
    df.columns = [c.strip() for c in df.columns]
    rename_map = {
        "Cadastro": "cadastro",
        "Nome": "nome",
        "Filial": "filial",
        "Data": "data",
        "Hora": "hora",
        "Dir": "dir",
        "Tipo": "tipo",
        "Acesso": "acesso",
        "Planta": "planta",
        "Coletor": "coletor",
        "Descricao Coletor": "descricao_coletor",
        "Descrição Coletor": "descricao_coletor",
        "Uso Acesso": "uso_acesso",
        "Cargo": "cargo",
        "Desligado": "desligado",
        "Área": "area",
        "Area": "area",
        "Departamento": "departamento",
        "Heranca": "heranca",
        "Herança": "heranca",
        "Cargo Especifico": "cargo_especifico",
        "Cargo Específico": "cargo_especifico",
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

def _clean_date_time(chunk: pd.DataFrame) -> tuple[pd.Series, pd.Series, pd.Series]:
    # data_iso
    data_dt = pd.to_datetime(chunk["data"], errors="coerce")
    data_iso = pd.Series(data_dt.dt.strftime("%Y-%m-%d"), dtype="string")

    # hora_iso (extrai HH:MM(:SS)? e normaliza)
    hora_str = chunk["hora"].astype("string").str.strip()
    hora_extr = hora_str.str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?)", expand=False)
    hora_clean = hora_extr.where(
        hora_extr.str.contains(r":\d{2}:\d{2}$", regex=True),
        hora_extr + ":00",
    )
    hora_clean = hora_clean.str.replace(r"^(\d):", r"0\1:", regex=True)
    hora_dt = pd.to_datetime(hora_clean, format="%H:%M:%S", errors="coerce")
    hora_iso = pd.Series(hora_dt.dt.strftime("%H:%M:%S"), dtype="string")

    # ts_iso
    ts = pd.to_datetime((data_iso + " " + hora_iso), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    ts_iso = pd.Series(ts.dt.strftime("%Y-%m-%d %H:%M:%S"), dtype="string")

    return data_iso, hora_iso, ts_iso

def _normalize_cargo_series(series: pd.Series) -> pd.Series:
    src = series.astype("string").fillna("").str.strip()
    upper = src.str.upper()
    out = upper.str[:1]

    out = out.where(~(upper.str.startswith("E") | upper.str.contains("EXEC", na=False)), "E")
    out = out.where(~(upper.str.startswith("M") | upper.str.contains("MENS", na=False)), "M")

    horista_like = (
        upper.str.startswith("H")
        | upper.str.contains("HOR", na=False)
        | upper.str.startswith("T")
        | upper.str.contains("TERC", na=False)
        | upper.str.contains("THIRD", na=False)
        | upper.str.contains("OUTSOUR", na=False)
        | upper.str.contains("3R", na=False)
    )
    out = out.where(~horista_like, "H")
    return out.fillna("").astype(str)

# =========================
# HEADCOUNT (HR) - helpers (roll-up por Heranca)
# =========================
def _hc_norm(x):
    return ("" if pd.isna(x) else str(x).strip().upper())

def _hc_to_dt(s):
    return pd.to_datetime(s, errors="coerce", dayfirst=True)

def _hc_rename_heranca(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df, pd.DataFrame) and "Herança" in df.columns and "Heranca" not in df.columns:
        return df.rename(columns={"Herança": "Heranca"})
    return df

def _hc_area_clean(area: str) -> str:
    # remove tudo que não é letra/número -> base p/ prefixo
    return re.sub(r"[^A-Z0-9]", "", _hc_norm(area))

def _hc_map_area_to_heranca(df_ativos: pd.DataFrame, df_desligados: pd.DataFrame) -> dict:
    frames = []
    for df in (_hc_rename_heranca(df_ativos), _hc_rename_heranca(df_desligados)):
        if isinstance(df, pd.DataFrame) and {"Área", "Heranca"}.issubset(df.columns):
            t = df[["Área", "Heranca"]].copy()
            t["Área"] = t["Área"].map(_hc_norm)
            t["Heranca"] = t["Heranca"].map(_hc_norm)
            t = t[(t["Área"] != "") & (t["Heranca"] != "") & (t["Heranca"] != "****")]
            frames.append(t)
    if not frames:
        return {}
    cat = pd.concat(frames, ignore_index=True)
    # moda por área (estável o suficiente)
    return cat.groupby("Área")["Heranca"].agg(
        lambda s: s.mode().iat[0] if not s.mode().empty else ""
    ).to_dict()

def _hc_area_parents(area: str, areaher: dict) -> list[str]:
    area_n = _hc_norm(area)
    clean = _hc_area_clean(area_n)
    her = _hc_norm(areaher.get(area_n, ""))
    if not clean or not her:
        return []

    parents = []
    for cand, h in (areaher or {}).items():
        if _hc_norm(h) != her:
            continue
        cand_n = _hc_norm(cand)
        cand_clean = _hc_area_clean(cand_n)
        if len(cand_clean) < len(clean) and clean.startswith(cand_clean):
            parents.append(cand_n)

    # mais próximo primeiro
    parents = sorted(set(parents), key=lambda a: len(_hc_area_clean(a)), reverse=True)
    return parents

def _hc_rollup_full_counts(full_counts: dict, areaher: dict) -> dict:
    out = dict(full_counts or {})
    for a_full, n in (full_counts or {}).items():
        a_full_n = _hc_norm(a_full)
        for p in _hc_area_parents(a_full_n, areaher):
            out[p] = out.get(p, 0) + int(n)
    return out

def _hc_mov_id_col(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    norm = {c: _hc_norm(c) for c in cols}
    exact = {"CADASTRO", "Nº PESSOAL", "NO PESSOAL", "NUM PESSOAL", "ID", "MATRICULA", "MATRÍCULA"}
    for c, n in norm.items():
        if n in exact:
            return c
    for c, n in norm.items():
        if any(k in n for k in ["CADASTRO", "PESSOAL", "MATRIC"]):
            return c
    raise KeyError(f"Não encontrei coluna de ID em df_movimentacoes. Colunas: {cols}")

def _hc_consolida_bases(df_ativos: pd.DataFrame, df_desligados: pd.DataFrame) -> pd.DataFrame:
    # df_ativos (snapshot atual)
    dep = _hc_rename_heranca(df_ativos).copy()
    if not dep.empty:
        dep["Cadastro"] = dep["Cadastro"].astype(str)
        dep["Área"] = dep.get("Área", "").map(_hc_norm) if "Área" in dep.columns else ""
        dep["Heranca"] = dep.get("Heranca", "").map(_hc_norm) if "Heranca" in dep.columns else ""
        dep["Data_Admissão"] = _hc_to_dt(dep.get("Data_Admissão")) if "Data_Admissão" in dep.columns else pd.NaT

        dep_cur = (
            dep.sort_values(["Cadastro", "Data_Admissão"])
               .drop_duplicates("Cadastro", keep="last")
               .rename(columns={"Área": "Área_cur", "Heranca": "Her_cur", "Data_Admissão": "Adm_dep"})
        )[["Cadastro", "Área_cur", "Her_cur", "Adm_dep"]]
    else:
        dep_cur = pd.DataFrame(columns=["Cadastro", "Área_cur", "Her_cur", "Adm_dep"])

    # df_desligados (histórico)
    afa = _hc_rename_heranca(df_desligados).copy()
    if not afa.empty:
        afa["Cadastro"] = afa["Cadastro"].astype(str)
        afa["Área"] = afa.get("Área", "").map(_hc_norm) if "Área" in afa.columns else ""
        afa["Heranca"] = afa.get("Heranca", "").map(_hc_norm) if "Heranca" in afa.columns else ""
        afa["Data_Adm"] = _hc_to_dt(afa.get("Data_Adm")) if "Data_Adm" in afa.columns else pd.NaT
        afa["Data_Dem"] = _hc_to_dt(afa.get("Data_Dem")) if "Data_Dem" in afa.columns else pd.NaT

        # primeira admissão
        adm_first = (
            afa.sort_values(["Cadastro", "Data_Adm"])
               .dropna(subset=["Data_Adm"])
               .drop_duplicates("Cadastro", keep="first")
               .rename(columns={"Área": "Área_on_adm", "Heranca": "Her_on_adm", "Data_Adm": "Adm_first"})
        )[["Cadastro", "Área_on_adm", "Her_on_adm", "Adm_first"]]

        # última demissão
        dem_last = (
            afa.sort_values(["Cadastro", "Data_Dem"])
               .dropna(subset=["Data_Dem"])
               .drop_duplicates("Cadastro", keep="last")
               .rename(columns={"Data_Dem": "Dem_last"})
        )[["Cadastro", "Dem_last"]]
    else:
        adm_first = pd.DataFrame(columns=["Cadastro", "Área_on_adm", "Her_on_adm", "Adm_first"])
        dem_last = pd.DataFrame(columns=["Cadastro", "Dem_last"])

    base = dep_cur.merge(adm_first, on="Cadastro", how="outer").merge(dem_last, on="Cadastro", how="outer")

    base["Adm_global"] = base[["Adm_dep", "Adm_first"]].min(axis=1)
    base["Dem_global"] = base["Dem_last"]

    base["Área_init"] = base["Área_on_adm"].where(
        base["Área_on_adm"].notna() & (base["Área_on_adm"] != ""),
        base["Área_cur"],
    )

    return base

def _hc_area_asof(dt: pd.Timestamp, base: pd.DataFrame, df_movimentacoes: pd.DataFrame | None) -> pd.Series:
    # filtra ativos na data
    asof = base[["Cadastro", "Área_init", "Adm_global", "Dem_global"]].copy()
    asof = asof[
        (asof["Adm_global"].notna())
        & (asof["Adm_global"] <= dt)
        & (asof["Dem_global"].isna() | (asof["Dem_global"] > dt))
    ].copy()

    if df_movimentacoes is None or not isinstance(df_movimentacoes, pd.DataFrame) or df_movimentacoes.empty:
        return asof.set_index("Cadastro")["Área_init"].astype(str).map(_hc_norm)

    mv = df_movimentacoes.copy()
    idcol = _hc_mov_id_col(mv)
    mv[idcol] = mv[idcol].astype(str)

    # normaliza
    for c in ("Abreviação UO", "Abreviação UO antiga"):
        if c in mv.columns:
            mv[c] = mv[c].map(_hc_norm)
    mv["Data de alteração"] = _hc_to_dt(mv.get("Data de alteração"))

    mv = mv[
        mv["Data de alteração"].notna()
        & (mv.get("Abreviação UO", "") != "")
        & (mv.get("Abreviação UO antiga", "") != "")
    ].sort_values([idcol, "Data de alteração"])

    # origem histórica: primeira área antiga
    earliest_old = mv.groupby(idcol)["Abreviação UO antiga"].first().rename("Área_earliest_old")

    # posição vigente: última área nova até a data
    mv_upto = mv[mv["Data de alteração"] <= dt]
    last_new = mv_upto.groupby(idcol)["Abreviação UO"].last().rename("Área_last_upto")

    eff = asof.merge(earliest_old, left_on="Cadastro", right_index=True, how="left")
    eff = eff.merge(last_new, left_on="Cadastro", right_index=True, how="left")

    eff["Área_asof"] = eff["Área_last_upto"].where(
        eff["Área_last_upto"].notna() & (eff["Área_last_upto"] != ""),
        eff["Área_earliest_old"].where(
            eff["Área_earliest_old"].notna() & (eff["Área_earliest_old"] != ""),
            eff["Área_init"],
        ),
    )

    return eff.set_index("Cadastro")["Área_asof"].astype(str).map(_hc_norm)

def _hc_snapshot_from_df_ativos(df_ativos: pd.DataFrame, areaher: dict) -> pd.Series:
    dep = _hc_rename_heranca(df_ativos).copy()
    if dep.empty or "Área" not in dep.columns or "Cadastro" not in dep.columns:
        return pd.Series(dtype=int)

    dep["Cadastro"] = dep["Cadastro"].astype(str)
    dep["Área"] = dep["Área"].map(_hc_norm)
    dep = dep[dep["Área"] != ""].copy()

    if "Data_Admissão" in dep.columns:
        dep["Data_Admissão"] = _hc_to_dt(dep["Data_Admissão"])
        dep = dep.sort_values(["Cadastro", "Data_Admissão"]).drop_duplicates("Cadastro", keep="last")
    else:
        dep = dep.drop_duplicates("Cadastro", keep="last")

    full_counts = dep["Área"].value_counts().to_dict()
    rolled = _hc_rollup_full_counts(full_counts, areaher)
    return pd.Series(rolled, dtype=int)

def compute_df_headcount_from_hr(
    df_ativos: pd.DataFrame,
    df_desligados: pd.DataFrame,
    df_movimentacoes: pd.DataFrame | None,
    anos=(2023, 2024, 2025),
) -> pd.DataFrame:
    # mapa área -> herança (pra roll-up)
    areaher = _hc_map_area_to_heranca(df_ativos, df_desligados)

    # base consolidada
    base = _hc_consolida_bases(df_ativos, df_desligados)

    # headcount por ano via snapshot 31/12 (as-of) + roll-up
    hc_year = {}
    for y in anos:
        dt = pd.Timestamp(int(y), 12, 31)
        area_asof = _hc_area_asof(dt, base, df_movimentacoes).dropna().astype(str).map(_hc_norm)
        full_counts = area_asof.value_counts().to_dict()
        hc_year[int(y)] = pd.Series(_hc_rollup_full_counts(full_counts, areaher), dtype=int)

    # ancoragem opcional do último ano via df_ativos (mantém sua lógica)
    if anos:
        last_year = int(max(anos))
        snap = _hc_snapshot_from_df_ativos(df_ativos, areaher)
        if isinstance(snap, pd.Series) and not snap.empty:
            hc_year[last_year] = snap.astype(int)

    # catálogo final de áreas (inclui subáreas e pais)
    all_areas = set()
    for s in hc_year.values():
        all_areas |= set(map(str, s.index))
    all_areas |= set(map(str, (areaher or {}).keys()))
    # garante pais no catálogo
    for a in list(all_areas):
        for p in _hc_area_parents(a, areaher):
            all_areas.add(p)

    areas = sorted({_hc_norm(a) for a in all_areas if _hc_norm(a)})

    # monta tabela area x ano
    out = pd.DataFrame({"area": areas})
    for y in (2023, 2024, 2025):
        s = hc_year.get(y, pd.Series(dtype=int)).reindex(areas).fillna(0).astype(int)
        out[f"y{y}"] = s.values

    return out[["area", "y2023", "y2024", "y2025"]]

# =========================
# Headcount & Turnover (compatível com o app antigo)
# =========================
# ------------------------------------------------------------
# Headcount & Turnover por área/ano com roll-up condicionado
# Regra:
#   HC → foto em 31/12 de cada ano (FULL + pais/avós na mesma Heranca).
#   E  → pessoas que entraram na área no ano (admissão + transferência).
#   S  → pessoas que saíram da área no ano (demissão + transferência).
#
# Transferências entre subáreas da MESMA área macro
# (ex.: E-IT ↔ E-IE) e MESMA Heranca:
#   - contam em E/S das subáreas;
#   - NÃO sobem para a área macro (linha geral), exceto quando a
#     própria área macro é origem ou destino (pai <-> filho).
# ------------------------------------------------------------

_norm = lambda x: ("" if pd.isna(x) else str(x).strip().upper())

# ... TODO O BLOCO HT (SEM ALTERAÇÕES) ...
# (mantido exatamente igual ao que você colou; não repeti aqui para não duplicar,
#  já que sua solicitação foi “alterar apenas o necessário”.)

# =========================
# DDL (MATERIALIZADO)
# =========================
DDL_SQL = r"""
PRAGMA foreign_keys = ON;

-- Base (flat)
DROP TABLE IF EXISTS evento_acesso_flat;
CREATE TABLE evento_acesso_flat (
  cadastro            TEXT,
  data                TEXT,  -- YYYY-MM-DD
  hora                TEXT,  -- HH:MM:SS
  ts                  TEXT,  -- YYYY-MM-DD HH:MM:SS
  filial              TEXT,
  planta              TEXT,
  coletor             TEXT,  -- UPPER no ETL
  descricao_coletor   TEXT,
  dir                 TEXT,  -- UPPER no ETL
  tipo                TEXT,
  acesso              TEXT,
  uso_acesso          TEXT
);

-- Dim colaborador (último estado por cadastro)
DROP TABLE IF EXISTS dim_colaborador;
CREATE TABLE dim_colaborador (
  cadastro            TEXT PRIMARY KEY,
  nome                TEXT,
  cargo               TEXT,
  cargo_especifico    TEXT,
  area                TEXT,
  departamento        TEXT,
  heranca             TEXT,
  desligado           TEXT,
  created_at          TEXT DEFAULT (datetime('now'))
) WITHOUT ROWID;

-- Dim data (calendário)
DROP TABLE IF EXISTS dim_data;
CREATE TABLE dim_data (
  data TEXT PRIMARY KEY
) WITHOUT ROWID;

-- Almoço clusterizado (tokens por evento)
DROP TABLE IF EXISTS agg_almoco_cluster;
CREATE TABLE agg_almoco_cluster (
  cadastro TEXT,
  data     TEXT,
  cluster_token TEXT,
  PRIMARY KEY (cadastro, data, cluster_token)
) WITHOUT ROWID;

-- Agregados p/ app
DROP TABLE IF EXISTS agg_diarios_unico;
CREATE TABLE agg_diarios_unico (
  cadastro TEXT,
  data     TEXT,
  planta   TEXT,
  PRIMARY KEY (cadastro, data)
) WITHOUT ROWID;

DROP TABLE IF EXISTS agg_registros_semanais;
CREATE TABLE agg_registros_semanais (
  cadastro TEXT,
  ano      INTEGER,
  semana_por_ano INTEGER,
  data     TEXT,
  contagem_presenca INTEGER,
  PRIMARY KEY (cadastro, ano, semana_por_ano)
) WITHOUT ROWID;

DROP VIEW IF EXISTS agg_semana_enriquecida;
DROP TABLE IF EXISTS agg_semana_enriquecida;
CREATE VIEW agg_semana_enriquecida AS
SELECT
  s.cadastro,
  c.nome,
  c.cargo,
  c.cargo_especifico,
  c.area,
  c.departamento,
  c.heranca,
  c.desligado,
  s.ano,
  s.semana_por_ano,
  s.data,
  s.contagem_presenca
FROM agg_registros_semanais s
LEFT JOIN dim_colaborador c
  ON c.cadastro = s.cadastro;

DROP TABLE IF EXISTS agg_almoco;
CREATE TABLE agg_almoco (
  cadastro TEXT,
  data     TEXT,
  hora     TEXT,
  dir      TEXT,
  planta   TEXT,
  coletor  TEXT,
  descricao_coletor TEXT
);

-- Opcionais
DROP TABLE IF EXISTS gptw;
CREATE TABLE gptw (area TEXT, y2025 REAL, y2024 REAL, y2023 REAL);

DROP TABLE IF EXISTS movimentacoes;
CREATE TABLE movimentacoes (
  cadastro TEXT,
  abreviacao_uo_antiga TEXT,
  desc_uo_antiga TEXT,
  desc_cargo_antigo TEXT,
  abreviacao_uo TEXT,
  desc_uo TEXT,
  desc_cargo TEXT,
  tipo_movimentacao TEXT,
  data_alteracao TEXT
);

DROP TABLE IF EXISTS headcount;
CREATE TABLE headcount (
  area TEXT PRIMARY KEY,
  heranca TEXT,
  y2023_hc INTEGER, y2023_e INTEGER, y2023_s INTEGER,
  y2024_hc INTEGER, y2024_e INTEGER, y2024_s INTEGER,
  y2025_hc INTEGER, y2025_e INTEGER, y2025_s INTEGER
) WITHOUT ROWID;

-- =========================
-- NOVO (necessário): TENURE + EVENTOS EXTERNOS (Fora da VW)
-- =========================
DROP TABLE IF EXISTS colaborador_hr;
CREATE TABLE colaborador_hr (
  cadastro  TEXT PRIMARY KEY,
  data_adm  TEXT,   -- YYYY-MM-DD
  data_dem  TEXT    -- YYYY-MM-DD (NULL se ativo)
) WITHOUT ROWID;

DROP TABLE IF EXISTS eventos_rh;
CREATE TABLE eventos_rh (
  cadastro     TEXT,
  data_evento  TEXT,   -- YYYY-MM-DD
  de_uo        TEXT,
  para_uo      TEXT,
  tipo_evento  TEXT
);

CREATE INDEX IF NOT EXISTS ix_ev_rh_data ON eventos_rh(data_evento);
CREATE INDEX IF NOT EXISTS ix_ev_rh_cad  ON eventos_rh(cadastro);

-- Views PT-BR (contratos do app)
DROP VIEW IF EXISTS vw_colaboradores;
CREATE VIEW vw_colaboradores AS
SELECT
  cadastro            AS "Cadastro",
  nome                AS "Nome",
  cargo               AS "Cargo",
  cargo_especifico    AS "Cargo Especifico",
  area                AS "Área",
  departamento        AS "Departamento",
  heranca             AS "Heranca",
  desligado           AS "Desligado"
FROM dim_colaborador;

DROP VIEW IF EXISTS vw_df_diarios_unico;
CREATE VIEW vw_df_diarios_unico AS
SELECT cadastro AS "Cadastro", planta AS "Planta", data AS "Data"
FROM agg_diarios_unico;

DROP VIEW IF EXISTS vw_df_diarios;
CREATE VIEW vw_df_diarios AS
SELECT cadastro AS "Cadastro", planta AS "Planta", data AS "Data"
FROM agg_diarios_unico;

DROP VIEW IF EXISTS vw_df_registros_semanais;
CREATE VIEW vw_df_registros_semanais AS
SELECT
  cadastro AS "Cadastro",
  ano AS "Ano",
  semana_por_ano AS "Semana por Ano",
  data AS "Data",
  contagem_presenca AS "Contagem Presença"
FROM agg_registros_semanais;

DROP VIEW IF EXISTS vw_semana_enriquecida;
CREATE VIEW vw_semana_enriquecida AS
SELECT
  cadastro AS "Cadastro",
  nome AS "Nome",
  cargo AS "Cargo",
  cargo_especifico AS "Cargo Especifico",
  area AS "Área",
  departamento AS "Departamento",
  heranca AS "Heranca",
  desligado AS "Desligado",
  ano AS "Ano",
  semana_por_ano AS "Semana por Ano",
  data AS "Data",
  contagem_presenca AS "Contagem Presença"
FROM agg_semana_enriquecida;

DROP VIEW IF EXISTS vw_df_almoco;
CREATE VIEW vw_df_almoco AS
SELECT
  cadastro AS "Cadastro",
  data AS "Data",
  hora AS "Hora",
  dir AS "Dir",
  planta AS "Planta",
  coletor AS "Coletor",
  descricao_coletor AS "Descricao Coletor"
FROM agg_almoco;

DROP VIEW IF EXISTS vw_almoco_cluster;
CREATE VIEW vw_almoco_cluster AS
SELECT cadastro AS "Cadastro", data AS "Data", cluster_token AS "Planta"
FROM agg_almoco_cluster;

DROP VIEW IF EXISTS vw_dim_data;
CREATE VIEW vw_dim_data AS
SELECT data AS "Data" FROM dim_data;

DROP VIEW IF EXISTS vw_gptw;
CREATE VIEW vw_gptw AS
SELECT area AS "Área", CAST(y2025 AS REAL) AS "2025", CAST(y2024 AS REAL) AS "2024", CAST(y2023 AS REAL) AS "2023"
FROM gptw;

DROP VIEW IF EXISTS vw_movimentacoes;
CREATE VIEW vw_movimentacoes AS
SELECT
  m.cadastro AS "Cadastro",
  COALESCE(c.nome, '') AS "Nome",
  m.abreviacao_uo_antiga AS "Abreviação UO antiga",
  m.desc_uo_antiga AS "Desc. UO antiga",
  m.desc_cargo_antigo AS "Desc. Cargo antigo",
  m.abreviacao_uo AS "Abreviação UO",
  m.desc_uo AS "Desc. UO",
  m.desc_cargo AS "Desc. Cargo",
  m.data_alteracao AS "Data de alteração"
FROM movimentacoes m
LEFT JOIN dim_colaborador c ON c.cadastro = m.cadastro;

DROP VIEW IF EXISTS vw_headcount;
CREATE VIEW vw_headcount AS
SELECT
  area AS "Área",
  heranca AS "Heranca",
  CAST(y2023_hc AS INTEGER) AS "2023 hc", CAST(y2023_e AS INTEGER) AS "2023 e", CAST(y2023_s AS INTEGER) AS "2023 s",
  CAST(y2024_hc AS INTEGER) AS "2024 hc", CAST(y2024_e AS INTEGER) AS "2024 e", CAST(y2024_s AS INTEGER) AS "2024 s",
  CAST(y2025_hc AS INTEGER) AS "2025 hc", CAST(y2025_e AS INTEGER) AS "2025 e", CAST(y2025_s AS INTEGER) AS "2025 s"
FROM headcount;

-- =========================
-- NOVO: Views p/ TENURE e Eventos "Fora da VW"
-- =========================
DROP VIEW IF EXISTS vw_colaborador_hr;
CREATE VIEW vw_colaborador_hr AS
SELECT
  cadastro AS "Cadastro",
  data_adm AS "Data Adm",
  data_dem AS "Data Dem",
  CAST((julianday(COALESCE(data_dem, date('now','localtime'))) - julianday(data_adm)) AS INTEGER) AS "Dias Empresa"
FROM colaborador_hr;

DROP VIEW IF EXISTS vw_eventos_rh;
CREATE VIEW vw_eventos_rh AS
SELECT
  cadastro AS "Cadastro",
  data_evento AS "Data",
  de_uo AS "De",
  para_uo AS "Para",
  tipo_evento AS "Tipo"
FROM eventos_rh;
"""

# =========================
# EXECUÇÃO (SEM FUNÇÃO)
# =========================

# Espera-se que df_ronda exista no ambiente.
# ---- PATCH: df_headcount agora é OPCIONAL ----
req = ["df_ronda", "df_movimentacoes", "df_gptw", "df_ativos", "df_desligados"]
for name in req:
    assert name in globals(), f"{name} não encontrado no ambiente."
    assert isinstance(globals()[name], pd.DataFrame), f"{name} não é DataFrame."
    assert not globals()[name].empty, f"{name} está vazio."

df_ronda = globals()["df_ronda"]
df_movimentacoes = globals()["df_movimentacoes"]
df_gptw = globals()["df_gptw"]
df_ativos = globals()["df_ativos"]
df_desligados = globals()["df_desligados"]

df_headcount = globals().get("df_headcount", pd.DataFrame())
if not isinstance(df_headcount, pd.DataFrame):
    df_headcount = pd.DataFrame()
# ---------------------------------------------

assert "df_ronda" in globals(), "df_ronda não encontrado no ambiente. Carregue df_ronda antes de rodar o ETL."
df_ronda = globals()["df_ronda"]

SQLITE_PATH.parent.mkdir(parents=True, exist_ok=True)
if SQLITE_PATH.exists():
    SQLITE_PATH.unlink()

normalize_columns_inplace(df_ronda)

required = [
    "cadastro", "nome", "filial", "data", "hora", "dir", "tipo", "acesso", "planta",
    "coletor", "descricao_coletor", "uso_acesso", "cargo", "desligado", "area",
    "departamento", "heranca", "cargo_especifico",
]

# Para DIM: manter “último estado” por cadastro sem guardar 35M linhas em staging
# Guardamos em dict: cadastro -> (ts, nome, cargo, cargo_especifico, area, departamento, heranca, desligado)
dim_latest: dict[str, tuple[str, str, str, str, str, str, str, str]] = {}

conn = sqlite3.connect(SQLITE_PATH)
try:
    # Bulk load PRAGMAs
    conn.execute("PRAGMA page_size = 32768;")
    conn.execute("PRAGMA journal_mode = OFF;")
    conn.execute("PRAGMA synchronous = OFF;")
    conn.execute("PRAGMA temp_store = MEMORY;")
    conn.execute("PRAGMA cache_size = -600000;")  # ~600MB
    conn.execute("PRAGMA locking_mode = EXCLUSIVE;")
    conn.execute("PRAGMA foreign_keys = ON;")

    conn.executescript(DDL_SQL)
    cur = conn.cursor()

    insert_flat = """INSERT INTO evento_acesso_flat (
        cadastro,data,hora,ts,filial,planta,coletor,descricao_coletor,dir,tipo,acesso,uso_acesso
    ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?);"""

    n = len(df_ronda)
    for start in range(0, n, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, n)
        chunk = df_ronda.iloc[start:end].reindex(columns=required, fill_value=pd.NA).copy()
        chunk.replace({"nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "": pd.NA}, inplace=True)

        data_iso, hora_iso, ts_iso = _clean_date_time(chunk)

        # Flat (mínimo necessário p/ agregações + filtros)
        flat_df = pd.DataFrame({
            "cadastro": chunk["cadastro"].astype("string").str.strip(),
            "data": data_iso,
            "hora": hora_iso,
            "ts": ts_iso,
            "filial": chunk["filial"].astype("string").str.strip(),
            "planta": chunk["planta"].astype("string").str.strip(),
            "coletor": chunk["coletor"].astype("string").str.strip().str.upper(),
            "descricao_coletor": chunk["descricao_coletor"].astype("string").str.strip(),
            "dir": chunk["dir"].astype("string").str.strip().str.upper(),
            "tipo": chunk["tipo"].astype("string").str.strip(),
            "acesso": chunk["acesso"].astype("string").str.strip(),
            "uso_acesso": chunk["uso_acesso"].astype("string").str.strip(),
        })

        # ✅ bind-safe: pandas.NA -> None
        flat_obj = flat_df.astype("object").where(flat_df.notna(), None)

        conn.execute("BEGIN;")
        cur.executemany(insert_flat, flat_obj.itertuples(index=False, name=None))
        conn.commit()

        # Atualiza DIM (último estado por cadastro) usando ts
        # Faz isso em pandas para evitar staging gigante + window function em 35M.
        cargo_norm = _normalize_cargo_series(chunk["cargo"])
        cargo_esp = chunk["cargo_especifico"].astype("string").str.strip()
        cargo_esp = cargo_esp.where(
            ~cargo_esp.fillna("").eq(""),
            chunk["cargo"].astype("string").str.strip(),
        )
        cargo_norm_fallback = _normalize_cargo_series(cargo_esp)
        cargo_norm = cargo_norm.where(~cargo_norm.astype(str).str.strip().eq(""), cargo_norm_fallback)

        dim_chunk = pd.DataFrame({
            "cadastro": chunk["cadastro"].astype("string").str.strip(),
            "ts": ts_iso,
            "nome": chunk["nome"].astype("string").str.strip(),
            "cargo": cargo_norm,
            "cargo_especifico": cargo_esp,
            "area": chunk["area"].astype("string").str.strip(),
            "departamento": chunk["departamento"].astype("string").str.strip(),
            "heranca": chunk["heranca"].astype("string").str.strip(),
            "desligado": chunk["desligado"].astype("string").str.strip(),
        }).astype("object").where(lambda d: d.notna(), None)

        # remove cadastros/ts vazios
        dim_chunk = dim_chunk[(dim_chunk["cadastro"].notna()) & (dim_chunk["ts"].notna())]

        # pega o registro de maior ts por cadastro no chunk (ts ISO ordena como string)
        dim_chunk.sort_values(["cadastro", "ts"], inplace=True)
        last_per_cad = dim_chunk.groupby("cadastro", as_index=False).tail(1)

        for row in last_per_cad.itertuples(index=False):
            cad = row.cadastro
            ts = row.ts
            prev = dim_latest.get(cad)
            if prev is None or ts > prev[0]:
                dim_latest[cad] = (ts, row.nome, row.cargo, row.cargo_especifico, row.area, row.departamento, row.heranca, row.desligado)

    # Índices no flat (acelera agregações e filtros)
    conn.executescript("""
        CREATE INDEX IF NOT EXISTS ix_flat_cad_data_ts   ON evento_acesso_flat (cadastro, data, ts);
        CREATE INDEX IF NOT EXISTS ix_flat_data          ON evento_acesso_flat (data);
        CREATE INDEX IF NOT EXISTS ix_flat_planta        ON evento_acesso_flat (planta);
        CREATE INDEX IF NOT EXISTS ix_flat_tipo          ON evento_acesso_flat (tipo);
        CREATE INDEX IF NOT EXISTS ix_flat_dir           ON evento_acesso_flat (dir);
        CREATE INDEX IF NOT EXISTS ix_flat_coletor       ON evento_acesso_flat (coletor);
    """)

    # =========================
    # DIM_DATA (calendário)
    # =========================
    conn.executescript("""
        DELETE FROM dim_data;
        INSERT OR IGNORE INTO dim_data(data)
        WITH RECURSIVE dates(d) AS (
          SELECT (SELECT MIN(data) FROM evento_acesso_flat WHERE data IS NOT NULL)
          UNION ALL
          SELECT date(d, '+1 day') FROM dates
          WHERE d < (SELECT MAX(data) FROM evento_acesso_flat WHERE data IS NOT NULL)
        )
        SELECT d FROM dates;
    """)

    # Carrega DIM no SQLite
    conn.execute("BEGIN;")
    conn.execute("DELETE FROM dim_colaborador;")
    conn.executemany(
        "INSERT INTO dim_colaborador(cadastro,nome,cargo,cargo_especifico,area,departamento,heranca,desligado) VALUES (?,?,?,?,?,?,?,?);",
        [(cad, nome, cargo, cargo_esp, area, depto, her, desl) for cad, (ts, nome, cargo, cargo_esp, area, depto, her, desl) in dim_latest.items()]
    )
    conn.commit()
    conn.executescript("""
        CREATE INDEX IF NOT EXISTS ix_dim_area   ON dim_colaborador (area);
        CREATE INDEX IF NOT EXISTS ix_dim_depto  ON dim_colaborador (departamento);
        CREATE INDEX IF NOT EXISTS ix_dim_heranca ON dim_colaborador (heranca);
    """)

    # =========================
    # AGREGAÇÕES MATERIALIZADAS
    # =========================

    # Presença válida + diários único (último ts do dia por cadastro)
    conn.executescript("""
        DROP TABLE IF EXISTS tmp_presenca_validos;
        CREATE TEMP TABLE tmp_presenca_validos AS
        SELECT
          cadastro,
          data AS data_dia,
          ts,
          planta
        FROM evento_acesso_flat
        WHERE
          cadastro IS NOT NULL
          AND data IS NOT NULL
          AND ts IS NOT NULL
          AND planta IS NOT NULL
          AND planta <> '999'
          AND tipo IS NOT NULL
          AND tipo <> '0';

        CREATE INDEX IF NOT EXISTS ix_tmp_pres_cad_data_ts ON tmp_presenca_validos (cadastro, data_dia, ts);

        -- ts máximo por cadastro+dia
        DROP TABLE IF EXISTS tmp_last;
        DROP TABLE IF EXISTS tmp_last_rid;
        CREATE TEMP TABLE tmp_last AS
        SELECT cadastro, data_dia, MAX(ts) AS ts_max
        FROM tmp_presenca_validos
        GROUP BY cadastro, data_dia;

        CREATE INDEX IF NOT EXISTS ix_tmp_last ON tmp_last (cadastro, data_dia);

        -- Empate no mesmo ts: pega a MAIOR rowid (determinístico) e evita duplicar PK
        DROP TABLE IF EXISTS tmp_last_rid;
        CREATE TEMP TABLE tmp_last_rid AS
        SELECT
          p.cadastro,
          p.data_dia,
          MAX(p.rowid) AS rid
        FROM tmp_presenca_validos p
        JOIN tmp_last l
          ON p.cadastro = l.cadastro
         AND p.data_dia = l.data_dia
         AND p.ts = l.ts_max
        GROUP BY p.cadastro, p.data_dia;

        CREATE INDEX IF NOT EXISTS ix_tmp_last_rid ON tmp_last_rid (cadastro, data_dia);

        INSERT INTO agg_diarios_unico (cadastro, data, planta)
        SELECT
          p.cadastro,
          p.data_dia AS data,
          p.planta
        FROM tmp_presenca_validos p
        JOIN tmp_last_rid r
          ON p.cadastro = r.cadastro
         AND p.data_dia = r.data_dia
         AND p.rowid = r.rid;

        CREATE INDEX IF NOT EXISTS ix_diarios_data_planta_cad ON agg_diarios_unico (data, planta, cadastro);
    """)

    # Semana ISO (ancora quinta)
    conn.executescript("""
        WITH base AS (
          SELECT
            cadastro,
            planta,
            data AS data_dia,
            date(data, '-3 days', 'weekday 4') AS iso_anchor
          FROM agg_diarios_unico
        )
        INSERT INTO agg_registros_semanais (cadastro, ano, semana_por_ano, data, contagem_presenca)
        SELECT
          cadastro,
          CAST(strftime('%Y', iso_anchor) AS INTEGER) AS ano,
          CAST(strftime('%W', iso_anchor) AS INTEGER) + 1 AS semana_por_ano,
          date(iso_anchor, '-3 days') AS data,
          COUNT(*) AS contagem_presenca
        FROM base
        GROUP BY cadastro, ano, semana_por_ano, date(iso_anchor, '-3 days');

        CREATE INDEX IF NOT EXISTS ix_semana_data      ON agg_registros_semanais (data);
        CREATE INDEX IF NOT EXISTS ix_semana_cadastro  ON agg_registros_semanais (cadastro);
        CREATE INDEX IF NOT EXISTS ix_semana_ano_sem_cad ON agg_registros_semanais (ano, semana_por_ano, cadastro);
    """)

    # Semana enriquecida
    # (agg_semana_enriquecida é VIEW; não criar índices nela. Performance vem de índices nas tabelas base.)

    # Almoço (dir/coletor já UPPER)
    conn.executescript("""
        INSERT INTO agg_almoco (cadastro, data, hora, dir, planta, coletor, descricao_coletor)
        SELECT
          cadastro,
          data AS data,
          hora,
          dir,
          planta,
          coletor,
          descricao_coletor
        FROM evento_acesso_flat
        WHERE
          data IS NOT NULL
          AND hora IS NOT NULL
          AND (
            (
              dir = 'E'
              AND coletor LIKE '%REF%'
              AND coletor NOT LIKE '%GRILL%'
              AND coletor NOT LIKE '%GRIL%'
            )
            OR
            (
              dir = 'S'
              AND coletor LIKE '%PORT%'
              AND hora >= '11:00:00' AND hora <= '14:00:00'
            )
          );

        CREATE INDEX IF NOT EXISTS ix_almoco_cad_data          ON agg_almoco (cadastro, data);
        CREATE INDEX IF NOT EXISTS ix_almoco_data_planta_cad   ON agg_almoco (data, planta, cadastro);
        CREATE INDEX IF NOT EXISTS ix_almoco_data_coletor      ON agg_almoco (data, coletor);
    """)

    # =========================
    # ALMOÇO CLUSTERIZADO (tokens por janela de 3 minutos)
    # =========================
    conn.executescript("""
        DELETE FROM agg_almoco_cluster;

        INSERT INTO agg_almoco_cluster(cadastro, data, cluster_token)
        WITH base AS (
          SELECT
            cadastro,
            data,
            dir,
            hora,
            CAST(strftime('%s', data || ' ' || hora) AS INTEGER) AS ts
          FROM agg_almoco
          WHERE
            cadastro IS NOT NULL
            AND data IS NOT NULL
            AND hora IS NOT NULL
            AND dir IS NOT NULL
        ),
        ordered AS (
          SELECT
            *,
            LAG(ts) OVER (PARTITION BY data, dir ORDER BY ts) AS prev_ts
          FROM base
        ),
        flags AS (
          SELECT
            *,
            CASE WHEN prev_ts IS NULL OR (ts - prev_ts) > 180 THEN 1 ELSE 0 END AS new_grp
          FROM ordered
        ),
        clu AS (
          SELECT
            *,
            SUM(new_grp) OVER (PARTITION BY data, dir ORDER BY ts ROWS UNBOUNDED PRECEDING) AS grp
          FROM flags
        )
        SELECT
          cadastro,
          data,
          (dir || printf('%03d', grp)) AS cluster_token
        FROM clu
        GROUP BY cadastro, data, cluster_token;

        CREATE INDEX IF NOT EXISTS ix_almococl_data_token_cad ON agg_almoco_cluster (data, cluster_token, cadastro);
        CREATE INDEX IF NOT EXISTS ix_almococl_cad_data       ON agg_almoco_cluster (cadastro, data);
    """)

    # =========================
    # CARGAS OBRIGATÓRIOS
    # =========================
    g = df_gptw.copy()
    g.columns = [c.strip() for c in g.columns]
    if "Área" in g.columns:
        g.rename(columns={"Área": "area"}, inplace=True)
    for y in ("2025", "2024", "2023"):
        if y in g.columns:
            g[y] = pd.to_numeric(g[y].replace({"**": np.nan}), errors="coerce")
    g2 = g[["area", "2025", "2024", "2023"]].rename(columns={"2025": "y2025", "2024": "y2024", "2023": "y2023"})
    g2 = g2.astype("object").where(g2.notna(), None)
    conn.execute("DELETE FROM gptw;")
    conn.executemany("INSERT INTO gptw(area,y2025,y2024,y2023) VALUES (?,?,?,?);", g2.itertuples(index=False, name=None))

    m = df_movimentacoes.copy()
    m.columns = [c.strip() for c in m.columns]
    rename = {
        "Cadastro": "cadastro",
        "Abreviação UO antiga": "abreviacao_uo_antiga",
        "Desc. UO antiga": "desc_uo_antiga",
        "Desc. Cargo antigo": "desc_cargo_antigo",
        "Abreviação UO": "abreviacao_uo",
        "Desc. UO": "desc_uo",
        "Desc. Cargo": "desc_cargo",
        "Tipo da Movimentação": "tipo_movimentacao",
        "Data de alteração": "data_alteracao",
    }
    m.rename(columns={k: v for k, v in rename.items() if k in m.columns}, inplace=True)
    if "data_alteracao" in m.columns:
        m["data_alteracao"] = pd.to_datetime(m["data_alteracao"], errors="coerce").dt.strftime("%Y-%m-%d")

    cols = ["cadastro","abreviacao_uo_antiga","desc_uo_antiga","desc_cargo_antigo","abreviacao_uo","desc_uo","desc_cargo","tipo_movimentacao","data_alteracao"]
    for c in cols:
        if c not in m.columns:
            m[c] = pd.NA
    m2 = m[cols].astype("object").where(m[cols].notna(), None)

    conn.execute("DELETE FROM movimentacoes;")
    conn.executemany(
        """INSERT INTO movimentacoes(
            cadastro,abreviacao_uo_antiga,desc_uo_antiga,desc_cargo_antigo,
            abreviacao_uo,desc_uo,desc_cargo,tipo_movimentacao,data_alteracao
        ) VALUES (?,?,?,?,?,?,?,?,?);""",
        m2.itertuples(index=False, name=None),
    )
    conn.executescript("""
        CREATE INDEX IF NOT EXISTS ix_mov_cadastro ON movimentacoes(cadastro);
        CREATE INDEX IF NOT EXISTS ix_mov_data ON movimentacoes(data_alteracao);
    """)

    # =========================
    # NOVO (necessário): TENURE (idade de empresa) + EVENTOS EXTERNOS "Fora da VW"
    # =========================
    df_ativos_hr = globals().get("df_ativos", None)
    df_desligados_hr = globals().get("df_desligados", None)

    def _to_dt(s):
        return pd.to_datetime(s, errors="coerce", dayfirst=True)

    def _pick_col(df: pd.DataFrame, *cands: str) -> str | None:
        for c in cands:
            if c in df.columns:
                return c
        return None

    # zera sempre (DW pode existir sem RH)
    conn.execute("DELETE FROM colaborador_hr;")
    conn.execute("DELETE FROM eventos_rh;")

    if (isinstance(df_ativos_hr, pd.DataFrame) and not df_ativos_hr.empty) or (isinstance(df_desligados_hr, pd.DataFrame) and not df_desligados_hr.empty):
        # ---- colaborador_hr (tenure) ----
        adm = pd.Series(dtype="datetime64[ns]")
        dem = pd.Series(dtype="datetime64[ns]")

        if isinstance(df_desligados_hr, pd.DataFrame) and not df_desligados_hr.empty and "Cadastro" in df_desligados_hr.columns:
            t = df_desligados_hr.copy()
            t["Cadastro"] = t["Cadastro"].astype(str)

            c_adm = _pick_col(t, "Data_Adm", "Data Adm", "Data Admissão", "Data_Admissão")
            c_dem = _pick_col(t, "Data_Dem", "Data Dem", "Data Demissão", "Data_Demissão")

            if c_adm:
                t[c_adm] = _to_dt(t[c_adm])
                adm = t.dropna(subset=[c_adm]).groupby("Cadastro")[c_adm].min()
            if c_dem:
                t[c_dem] = _to_dt(t[c_dem])
                dem = t.dropna(subset=[c_dem]).groupby("Cadastro")[c_dem].max()

        if isinstance(df_ativos_hr, pd.DataFrame) and not df_ativos_hr.empty and "Cadastro" in df_ativos_hr.columns:
            t = df_ativos_hr.copy()
            t["Cadastro"] = t["Cadastro"].astype(str)
            c_adm = _pick_col(t, "Data_Admissão", "Data Admissão", "Data_Adm", "Data Adm")
            if c_adm:
                t[c_adm] = _to_dt(t[c_adm])
                adm2 = t.dropna(subset=[c_adm]).groupby("Cadastro")[c_adm].min()
                if not adm2.empty:
                    adm = pd.concat([adm, adm2], axis=0).groupby(level=0).min()

        hr = pd.DataFrame({"cadastro": adm.index.astype(str), "data_adm": adm.dt.strftime("%Y-%m-%d").values})
        hr = hr.merge(dem.rename("data_dem").dt.strftime("%Y-%m-%d"), left_on="cadastro", right_index=True, how="left")
        hr = hr.astype("object").where(hr.notna(), None)

        if not hr.empty:
            conn.executemany(
                "INSERT INTO colaborador_hr(cadastro,data_adm,data_dem) VALUES (?,?,?);",
                hr.itertuples(index=False, name=None),
            )

        # ---- eventos_rh (fora <-> vw) ----
        eventos = []

        # suporte: inferir área de entrada (UO Abreviado) via 1ª movimentação antiga, p/ quem só está em df_ativos
        orig_info = {}
        if isinstance(df_movimentacoes, pd.DataFrame) and not df_movimentacoes.empty:
            mv = df_movimentacoes.copy()
            idcol = _hc_mov_id_col(mv)
            mv[idcol] = mv[idcol].astype(str)

            c_dt = _pick_col(mv, "Data de alteração", "Data Alteracao", "Data_Alteracao")
            c_old = _pick_col(mv, "Abreviação UO antiga", "Abreviacao UO antiga", "Abreviação UO Antiga")

            if c_dt and c_old:
                mv[c_dt] = _to_dt(mv[c_dt])
                mv[c_old] = mv[c_old].astype(str).str.strip()
                mv = mv.dropna(subset=[c_dt]).sort_values([idcol, c_dt])
                first_mv = mv.groupby(idcol).first().reset_index()
                for r in first_mv.itertuples(index=False):
                    cad = str(getattr(r, idcol))
                    area_old = getattr(r, c_old, "")
                    dt_first = getattr(r, c_dt, pd.NaT)
                    if isinstance(area_old, str) and area_old.strip() and pd.notna(dt_first):
                        orig_info[cad] = {"area_old": area_old.strip(), "dt_first": dt_first}

        cad_in_desl = set()
        # (A) df_desligados: gera eventos conforme datas (mantém múltiplos se existirem)
        if isinstance(df_desligados_hr, pd.DataFrame) and not df_desligados_hr.empty and "Cadastro" in df_desligados_hr.columns:
            t = df_desligados_hr.copy()
            t["Cadastro"] = t["Cadastro"].astype(str)
            cad_in_desl = set(t["Cadastro"].unique())

            c_area = _pick_col(t, "Área", "Area")
            c_adm = _pick_col(t, "Data_Adm", "Data Adm", "Data_Admissão", "Data Admissão")
            c_dem = _pick_col(t, "Data_Dem", "Data Dem", "Data Demissão", "Data_Demissão")

            if c_area:
                t[c_area] = t[c_area].astype(str).str.strip()

            if c_adm and c_area:
                t[c_adm] = _to_dt(t[c_adm])
                tt = t.dropna(subset=[c_adm])
                for r in tt.itertuples(index=False):
                    area = getattr(r, c_area, "")
                    dt = getattr(r, c_adm, pd.NaT)
                    if area and pd.notna(dt):
                        eventos.append((str(r.Cadastro), dt.strftime("%Y-%m-%d"), "Fora da VW", str(area), "Admissao"))

            if c_dem and c_area:
                t[c_dem] = _to_dt(t[c_dem])
                tt = t.dropna(subset=[c_dem])
                for r in tt.itertuples(index=False):
                    area = getattr(r, c_area, "")
                    dt = getattr(r, c_dem, pd.NaT)
                    if area and pd.notna(dt):
                        eventos.append((str(r.Cadastro), dt.strftime("%Y-%m-%d"), str(area), "Fora da VW", "Demissao"))

        # (B) df_ativos: admissões para quem não existe em df_desligados
        if isinstance(df_ativos_hr, pd.DataFrame) and not df_ativos_hr.empty and "Cadastro" in df_ativos_hr.columns:
            t = df_ativos_hr.copy()
            t["Cadastro"] = t["Cadastro"].astype(str)

            c_area = _pick_col(t, "Área", "Area")
            c_adm = _pick_col(t, "Data_Admissão", "Data Admissão", "Data_Adm", "Data Adm")

            if c_area:
                t[c_area] = t[c_area].astype(str).str.strip()

            if c_adm and c_area:
                t[c_adm] = _to_dt(t[c_adm])
                tt = t.dropna(subset=[c_adm])
                tt = tt[~tt["Cadastro"].isin(cad_in_desl)]

                for r in tt.itertuples(index=False):
                    cad = str(r.Cadastro)
                    dt_adm = getattr(r, c_adm, pd.NaT)
                    area_adm = str(getattr(r, c_area, "")).strip()

                    info = orig_info.get(cad)
                    if info and pd.notna(dt_adm) and pd.notna(info["dt_first"]) and dt_adm <= info["dt_first"]:
                        if info["area_old"]:
                            area_adm = info["area_old"]

                    if area_adm and pd.notna(dt_adm):
                        eventos.append((cad, dt_adm.strftime("%Y-%m-%d"), "Fora da VW", area_adm, "Admissao"))

        ev = pd.DataFrame(eventos, columns=["cadastro","data_evento","de_uo","para_uo","tipo_evento"]).drop_duplicates()
        ev = ev.astype("object").where(ev.notna(), None)

        if not ev.empty:
            conn.executemany(
                "INSERT INTO eventos_rh(cadastro,data_evento,de_uo,para_uo,tipo_evento) VALUES (?,?,?,?,?);",
                ev.itertuples(index=False, name=None),
            )

    # =========================
    # HEADCOUNT & TURNOVER (FEITO NO ETL)  ✅ CORRIGIDO
    # =========================
    anos = (2023, 2024, 2025)

    # (1) Herança por área
    areaher = _hc_map_area_to_heranca(df_ativos, df_desligados)

    # (2) Headcount (HC) por ano (preferencialmente calculado do RH; fallback para df_headcount)
    hc_df_calc = None
    try:
        hc_df_calc = compute_df_headcount_from_hr(df_ativos, df_desligados, df_movimentacoes, anos=anos)
    except Exception:
        hc_df_calc = None

    if isinstance(hc_df_calc, pd.DataFrame) and not hc_df_calc.empty and {"area", "y2023", "y2024", "y2025"}.issubset(hc_df_calc.columns):
        hc_base = hc_df_calc.copy()
        hc_base["area"] = hc_base["area"].map(_hc_norm)
        hc_base = hc_base.dropna(subset=["area"])
        hc_year = {
            2023: hc_base.set_index("area")["y2023"].fillna(0).astype(int),
            2024: hc_base.set_index("area")["y2024"].fillna(0).astype(int),
            2025: hc_base.set_index("area")["y2025"].fillna(0).astype(int),
        }
    else:
        # fallback: tenta usar df_headcount em formatos comuns
        h = df_headcount.copy()
        h.columns = [str(c).strip() for c in h.columns]

        # helper: limpa valores com setas/símbolos (ex.: '16▼', '0—', '20▲')
        def _hc_clean_int(v) -> int:
            if pd.isna(v):
                return 0
            s = str(v).strip()
            m = re.search(r"\d+", s)
            return int(m.group(0)) if m else 0

        if {"area", "y2023", "y2024", "y2025"}.issubset(set(map(str, h.columns))):
            # formato já normalizado (area + y2023/y2024/y2025)
            hb = h[["area", "y2023", "y2024", "y2025"]].copy()
            hb["area"] = hb["area"].map(_hc_norm)
            hb = hb.dropna(subset=["area"])
            hc_year = {
                2023: hb.set_index("area")["y2023"].fillna(0).astype(int),
                2024: hb.set_index("area")["y2024"].fillna(0).astype(int),
                2025: hb.set_index("area")["y2025"].fillna(0).astype(int),
            }
        else:
            # tenta detectar o formato do seu arquivo: Área | 2023 | E | S | 2024 | E | S | 2025 | E | S
            cols_stripped = [str(c).strip() for c in h.columns]
            has_year_cols = any(re.fullmatch(r"20\d{2}", c) for c in cols_stripped)

            if has_year_cols:
                hb = h.copy()
                # normaliza nome da coluna de área
                if "Área" in hb.columns:
                    hb = hb.rename(columns={"Área": "area"})
                elif "area" not in hb.columns:
                    hb = hb.rename(columns={hb.columns[0]: "area"})

                hb["area"] = hb["area"].map(_hc_norm)
                hb = hb.dropna(subset=["area"])

                # mapeia colunas por POSIÇÃO: [2023, E, S, 2024, E, S, ...]
                col_list = list(hb.columns)
                mapping = {}
                i = 0
                while i < len(col_list):
                    c = col_list[i]
                    cs = str(c).strip()
                    if cs.lower() == "area":
                        i += 1
                        continue
                    m = re.fullmatch(r"(20\d{2})", cs)
                    if m:
                        y = m.group(1)
                        mapping[c] = f"y{y}_hc"
                        if i + 1 < len(col_list):
                            mapping[col_list[i + 1]] = f"y{y}_e"
                        if i + 2 < len(col_list):
                            mapping[col_list[i + 2]] = f"y{y}_s"
                        i += 3
                    else:
                        i += 1

                hb = hb.rename(columns=mapping)

                # garante colunas esperadas e limpa números
                for y in (2023, 2024, 2025):
                    for suf in ("hc", "e", "s"):
                        col = f"y{y}_{suf}"
                        if col not in hb.columns:
                            hb[col] = 0
                        hb[col] = hb[col].map(_hc_clean_int).astype(int)

                # herança: se não vier no df_headcount, deriva do mapa areaher (área -> herança)
                if "heranca" not in hb.columns:
                    hb["heranca"] = hb["area"].map(lambda a: (areaher or {}).get(a, ""))
                    hb["heranca"] = hb["heranca"].fillna("").astype(str)

                hc_year = {
                    2023: hb.set_index("area")["y2023_hc"].fillna(0).astype(int),
                    2024: hb.set_index("area")["y2024_hc"].fillna(0).astype(int),
                    2025: hb.set_index("area")["y2025_hc"].fillna(0).astype(int),
                }
            else:
                # formato "app": Área + "2023 hc"...
                if "Área" in h.columns:
                    hb = h.rename(columns={"Área": "area"}).copy()
                else:
                    hb = h.copy()
                    hb["area"] = hb.get("area", pd.NA)
                hb["area"] = hb["area"].map(_hc_norm)

                def _pick_hc(col):
                    return col in hb.columns

                c23 = "2023 hc" if _pick_hc("2023 hc") else ("y2023" if _pick_hc("y2023") else None)
                c24 = "2024 hc" if _pick_hc("2024 hc") else ("y2024" if _pick_hc("y2024") else None)
                c25 = "2025 hc" if _pick_hc("2025 hc") else ("y2025" if _pick_hc("y2025") else None)

                for c in (c23, c24, c25):
                    if c:
                        hb[c] = hb[c].map(_hc_clean_int).astype(int)

                hc_year = {
                    2023: hb.set_index("area")[c23].fillna(0).astype(int) if c23 else pd.Series(dtype=int),
                    2024: hb.set_index("area")[c24].fillna(0).astype(int) if c24 else pd.Series(dtype=int),
                    2025: hb.set_index("area")[c25].fillna(0).astype(int) if c25 else pd.Series(dtype=int),
                }
    # (3) Preparação dos eventos (admissão/demissão + transferências)
    def _pick_col2(df: pd.DataFrame, *cands: str) -> str | None:
        for c in cands:
            if c in df.columns:
                return c
        return None

    # cache de pais (por área) e sets para "descendente"
    parents_cache: dict[str, list[str]] = {}
    parentset_cache: dict[str, set[str]] = {}

    def _parents(a: str) -> list[str]:
        a = _hc_norm(a)
        if a in parents_cache:
            return parents_cache[a]
        ps = _hc_area_parents(a, areaher)
        parents_cache[a] = ps
        parentset_cache[a] = set(ps)
        return ps

    def _is_desc(child: str | None, anc: str) -> bool:
        if child is None:
            return False
        c = _hc_norm(child)
        a = _hc_norm(anc)
        if c == "" or a == "":
            return False
        if c == a:
            return True
        _parents(c)
        return a in parentset_cache.get(c, set())

    # agregação como "pessoas" (sets) por (ano, área)
    entered: dict[tuple[int, str], set[str]] = defaultdict(set)
    left: dict[tuple[int, str], set[str]] = defaultdict(set)

    def _add_enter(cad: str, y: int, area: str) -> None:
        a = _hc_norm(area)
        if a:
            entered[(y, a)].add(cad)

    def _add_left(cad: str, y: int, area: str) -> None:
        a = _hc_norm(area)
        if a:
            left[(y, a)].add(cad)

    def _add_event_rollup_transfer(cad: str, dt: pd.Timestamp, old_area: str, new_area: str) -> None:
        if pd.isna(dt):
            return
        y = int(dt.year)
        if y not in anos:
            return
        o = _hc_norm(old_area)
        n = _hc_norm(new_area)
        if not o or not n or o == n:
            return

        # nível exato sempre conta
        _add_left(cad, y, o)
        _add_enter(cad, y, n)

        # sobe S para pais de origem somente se destino NÃO está no mesmo pai/avô
        for p in _parents(o):
            if _is_desc(n, p):
                continue
            _add_left(cad, y, p)

        # sobe E para pais de destino somente se origem NÃO está no mesmo pai/avô
        for p in _parents(n):
            if _is_desc(o, p):
                continue
            _add_enter(cad, y, p)

    def _add_event_rollup_inout(cad: str, dt: pd.Timestamp, area: str, is_enter: bool) -> None:
        if pd.isna(dt):
            return
        y = int(dt.year)
        if y not in anos:
            return
        a = _hc_norm(area)
        if not a:
            return
        if is_enter:
            _add_enter(cad, y, a)
            for p in _parents(a):
                _add_enter(cad, y, p)
        else:
            _add_left(cad, y, a)
            for p in _parents(a):
                _add_left(cad, y, p)

    # (3.1) Transferências (df_movimentacoes)
    mv_events = pd.DataFrame(columns=["cad", "dt", "old", "new"])
    if isinstance(df_movimentacoes, pd.DataFrame) and not df_movimentacoes.empty:
        mv = df_movimentacoes.copy()
        mv.columns = [c.strip() for c in mv.columns]
        idcol = _hc_mov_id_col(mv)
        mv[idcol] = mv[idcol].astype(str)

        c_dt = _pick_col2(mv, "Data de alteração", "Data Alteracao", "Data_Alteração", "Data_Alteracao", "data_alteracao", "Data", "DATA")
        c_old = _pick_col2(mv, "Abreviação UO antiga", "Abreviacao UO antiga", "abreviacao_uo_antiga", "De", "DE", "de", "de_uo")
        c_new = _pick_col2(mv, "Abreviação UO", "Abreviacao UO", "abreviacao_uo", "Para", "PARA", "para", "para_uo")

        if c_dt and c_old and c_new:
            mv["__dt"] = _to_dt(mv[c_dt])
            mv["__old"] = mv[c_old].map(_hc_norm)
            mv["__new"] = mv[c_new].map(_hc_norm)

            mv2 = mv.dropna(subset=["__dt"])
            mv2 = mv2[(mv2["__old"] != "") & (mv2["__new"] != "") & (mv2["__old"] != mv2["__new"])]

            mv_events = mv2[[idcol, "__dt", "__old", "__new"]].rename(
                columns={idcol: "cad", "__dt": "dt", "__old": "old", "__new": "new"}
            )

    if not mv_events.empty:
        for r in mv_events.itertuples(index=False):
            _add_event_rollup_transfer(str(r.cad), pd.Timestamp(r.dt), str(r.old), str(r.new))

    # (3.2) Admissões/Demissões (RH) com ajuste de área na admissão via 1ª mov. antiga
    base = _hc_consolida_bases(df_ativos, df_desligados)

    # mapa: 1ª mov (old) e data da 1ª mov, por cadastro
    first_old_by_cad = {}
    first_dt_by_cad = {}
    if not mv_events.empty:
        mv_sorted = mv_events.sort_values(["cad", "dt"])
        first = mv_sorted.groupby("cad", as_index=False).first()
        for r in first.itertuples(index=False):
            first_old_by_cad[str(r.cad)] = str(r.old)
            first_dt_by_cad[str(r.cad)] = pd.Timestamp(r.dt)

    # mapa: área em demissão (pega do df_desligados se tiver Data_Dem/Área)
    dem_area_from_desl = {}
    if isinstance(df_desligados, pd.DataFrame) and not df_desligados.empty:
        t = df_desligados.copy()
        t.columns = [c.strip() for c in t.columns]
        if "Cadastro" in t.columns and ("Área" in t.columns or "Area" in t.columns):
            c_area = "Área" if "Área" in t.columns else "Area"
            c_dem = _pick_col2(t, "Data_Dem", "Data Dem", "Data Demissão", "Data_Demissão")
            if c_dem:
                t["__dem"] = _to_dt(t[c_dem])
                t["__area"] = t[c_area].map(_hc_norm)
                t["Cadastro"] = t["Cadastro"].astype(str)
                t2 = t.dropna(subset=["__dem"]).sort_values(["Cadastro", "__dem"])
                last = t2.groupby("Cadastro", as_index=False).last()
                for r in last.itertuples(index=False):
                    if getattr(r, "__area", ""):
                        dem_area_from_desl[str(r.Cadastro)] = str(getattr(r, "__area"))

    # função: área vigente até uma data (usa última "new" <= dt; senão 1ª old; senão Área_init)
    def _area_upto(cad: str, dt: pd.Timestamp, area_init: str) -> str:
        cad = str(cad)
        if not mv_events.empty:
            # filtra só desse cad (custo aceitável por uso limitado: demissões)
            sub = mv_events[mv_events["cad"].astype(str) == cad]
            if not sub.empty:
                sub2 = sub[sub["dt"] <= dt]
                if not sub2.empty:
                    return _hc_norm(sub2.sort_values("dt").iloc[-1]["new"])
                # antes da 1ª mov, usa old
                return _hc_norm(sub.sort_values("dt").iloc[0]["old"])
        return _hc_norm(area_init)

    # admissões
    if not base.empty and {"Cadastro", "Adm_global", "Área_init"}.issubset(base.columns):
        for r in base.dropna(subset=["Adm_global"]).itertuples(index=False):
            cad = str(getattr(r, "Cadastro"))
            dt_adm = getattr(r, "Adm_global")
            if pd.isna(dt_adm):
                continue
            dt_adm = pd.Timestamp(dt_adm)
            area_adm = _hc_norm(getattr(r, "Área_init", ""))

            # ajuste: se admissão ocorreu antes (ou no dia) da 1ª movimentação, usa 1ª old
            fdt = first_dt_by_cad.get(cad)
            fold = first_old_by_cad.get(cad)
            if fold and fdt is not None and pd.notna(fdt) and dt_adm <= fdt:
                area_adm = _hc_norm(fold)

            _add_event_rollup_inout(cad, dt_adm, area_adm, is_enter=True)

    # demissões
    if not base.empty and {"Cadastro", "Dem_global", "Área_init"}.issubset(base.columns):
        for r in base.dropna(subset=["Dem_global"]).itertuples(index=False):
            cad = str(getattr(r, "Cadastro"))
            dt_dem = getattr(r, "Dem_global")
            if pd.isna(dt_dem):
                continue
            dt_dem = pd.Timestamp(dt_dem)

            # área na demissão: prioriza df_desligados; senão usa última mov <= dem; senão Área_init
            area_init = _hc_norm(getattr(r, "Área_init", ""))
            area_dem = _hc_norm(dem_area_from_desl.get(cad, "")) or _area_upto(cad, dt_dem, area_init)
            _add_event_rollup_inout(cad, dt_dem, area_dem, is_enter=False)

    # (4) Montagem final da tabela headcount (HC/E/S) + carga no SQLite
    all_areas = set()
    for y in anos:
        s = hc_year.get(y, pd.Series(dtype=int))
        all_areas |= set(map(str, s.index))
    all_areas |= {a for (_, a) in entered.keys()}
    all_areas |= {a for (_, a) in left.keys()}

    # garante pais no catálogo
    for a in list(all_areas):
        for p in _parents(a):
            all_areas.add(p)

    areas = sorted({_hc_norm(a) for a in all_areas if _hc_norm(a)})

    # herança: tenta preencher pais via filhos quando não existir no mapa
    her_map = { _hc_norm(k): _hc_norm(v) for k, v in (areaher or {}).items() if _hc_norm(k) and _hc_norm(v) and _hc_norm(v) != "****" }
    for child, hval in list(her_map.items()):
        for p in _parents(child):
            if p not in her_map:
                her_map[p] = hval

    rows = []
    for a in areas:
        rows.append((
            a,
            her_map.get(a, None),
            int(hc_year.get(2023, pd.Series(dtype=int)).get(a, 0)),
            int(len(entered.get((2023, a), set()))),
            int(len(left.get((2023, a), set()))),
            int(hc_year.get(2024, pd.Series(dtype=int)).get(a, 0)),
            int(len(entered.get((2024, a), set()))),
            int(len(left.get((2024, a), set()))),
            int(hc_year.get(2025, pd.Series(dtype=int)).get(a, 0)),
            int(len(entered.get((2025, a), set()))),
            int(len(left.get((2025, a), set()))),
        ))

    conn.execute("DELETE FROM headcount;")
    if rows:
        conn.executemany(
            """INSERT INTO headcount(
                area,heranca,
                y2023_hc,y2023_e,y2023_s,
                y2024_hc,y2024_e,y2024_s,
                y2025_hc,y2025_e,y2025_s
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?);""",
            rows,
        )
    conn.executescript("""
        CREATE INDEX IF NOT EXISTS ix_headcount_heranca ON headcount(heranca);
    """)

    # ---- PATCH: gera df_headcount + df_turnover em memória ----
    df_headcount_out = pd.DataFrame({
        "Área": [r[0] for r in rows],
        "Heranca": [r[1] for r in rows],
        "2023 hc": [r[2] for r in rows],
        "2023 e":  [r[3] for r in rows],
        "2023 s":  [r[4] for r in rows],
        "2024 hc": [r[5] for r in rows],
        "2024 e":  [r[6] for r in rows],
        "2024 s":  [r[7] for r in rows],
        "2025 hc": [r[8] for r in rows],
        "2025 e":  [r[9] for r in rows],
        "2025 s":  [r[10] for r in rows],
    })

    turnover_rows = []
    for r in rows:
        area = r[0]

        hc_2023, s_2023 = int(r[2]), int(r[4])
        hc_2024, s_2024 = int(r[5]), int(r[7])
        hc_2025, s_2025 = int(r[8]), int(r[10])

        hc_med_2023 = hc_2023
        hc_med_2024 = (hc_2023 + hc_2024) / 2.0
        hc_med_2025 = (hc_2024 + hc_2025) / 2.0

        def _to_pct(saidas, hc_med):
            if hc_med is None or hc_med <= 0:
                return 0.0
            return float(saidas) / float(hc_med) * 100.0

        turnover_rows.append((area, 2023, hc_2023, s_2023, hc_med_2023, _to_pct(s_2023, hc_med_2023)))
        turnover_rows.append((area, 2024, hc_2024, s_2024, hc_med_2024, _to_pct(s_2024, hc_med_2024)))
        turnover_rows.append((area, 2025, hc_2025, s_2025, hc_med_2025, _to_pct(s_2025, hc_med_2025)))

    df_turnover = pd.DataFrame(
        turnover_rows,
        columns=["area", "ano", "hc_fim_ano", "saidas", "hc_medio", "turnover_pct"],
    )

    globals()["df_headcount"] = df_headcount_out
    globals()["df_turnover"] = df_turnover
    # ---------------------------------------------------------

    # =========================
    # Limpeza + estatísticas
    # =========================
    conn.executescript("""
        DROP TABLE IF EXISTS tmp_presenca_validos;
        DROP TABLE IF EXISTS tmp_last;
        DROP TABLE IF EXISTS tmp_last_rid;
    """)

    if DO_ANALYZE:
        conn.execute("ANALYZE;")
    if DO_OPTIMIZE:
        try:
            conn.execute("PRAGMA optimize;")
        except Exception:
            pass
    if DO_VACUUM:
        try:
            conn.execute("VACUUM;")
        except Exception:
            pass

    # Finaliza banco para leitura (sem WAL)
    try:
        conn.execute("PRAGMA journal_mode = DELETE;")
        conn.execute("PRAGMA synchronous = NORMAL;")
    except Exception:
        pass
    conn.execute("PRAGMA locking_mode = NORMAL;")

finally:
    conn.close()

print("✅ DW gerado em:", SQLITE_PATH)

In [ ]:
# Visualizando a melhora do desempenho do painel (comparando o Parquet otimizado com o DW)
# ==========================================================
# CONFIG
# ==========================================================
PARQUET_PATH = Path(r'.\Documentação\Analisando Irregularidades dos Registros\df_ronda.parquet')
SQLITE_PATH  = Path(r'..\project\database\dw_ronda.sqlite')

PARQUET_PATH_ABS = PARQUET_PATH.resolve()
SQLITE_PATH_ABS  = SQLITE_PATH.resolve()

RANGE_MODE = "last_n_days"
N_DAYS = 90

# ==========================================================
# HELPERS
# ==========================================================
def connect_ro_sqlite(path: Path, immutable=True) -> sqlite3.Connection:
    path = path.resolve()  # absoluto só internamente
    if not path.exists():
        raise FileNotFoundError(f"DW não encontrado: {path.name}")
    uri = path.as_uri() + ("?mode=ro&immutable=1" if immutable else "?mode=ro")
    return sqlite3.connect(uri, uri=True)

def one(conn, sql: str, params=()):
    t0 = time.perf_counter()
    row = conn.execute(sql, params).fetchone()
    ms = (time.perf_counter() - t0) * 1000
    return ms, row

def exists_sqlite(conn, name: str, typ: str) -> bool:
    r = conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type=? AND name=? LIMIT 1",
        (typ, name)
    ).fetchone()
    return r is not None

def find_col_case_insensitive(cols, target_lower: str):
    mapping = {c.lower(): c for c in cols}
    return mapping.get(target_lower)

def pick_date_range_dw(conn) -> tuple[str, str]:
    # usa agg_diarios_unico como "verdade"
    dmin, dmax = conn.execute("SELECT MIN(data), MAX(data) FROM agg_diarios_unico").fetchone()
    if RANGE_MODE == "full":
        return dmin, dmax
    dmax_dt = pd.to_datetime(dmax)
    dmin_dt = dmax_dt - pd.Timedelta(days=N_DAYS - 1)
    return dmin_dt.strftime("%Y-%m-%d"), dmax_dt.strftime("%Y-%m-%d")

def parquet_size_mb(path: Path) -> float:
    return os.path.getsize(path) / (1024**2)

def sqlite_size_mb(path: Path, conn: sqlite3.Connection) -> float:
    page_count = conn.execute("PRAGMA page_count;").fetchone()[0]
    page_size  = conn.execute("PRAGMA page_size;").fetchone()[0]
    return (page_count * page_size) / (1024**2)

# ==========================================================
# 0) CHECKS
# ==========================================================
if not PARQUET_PATH.exists():
    raise FileNotFoundError(f"Parquet não encontrado: {PARQUET_PATH}")
if not SQLITE_PATH.exists():
    raise FileNotFoundError(f"DW não encontrado: {SQLITE_PATH}")

# tenta usar DuckDB (ideal para comparar Parquet sem carregar tudo em RAM)
try:
    import duckdb
    HAS_DUCKDB = True
except Exception:
    HAS_DUCKDB = False

# ==========================================================
# 1) CONECTA DW
# ==========================================================
conn = connect_ro_sqlite(SQLITE_PATH_ABS, immutable=True)
conn.execute("SELECT 1").fetchone()

d1, d2 = pick_date_range_dw(conn)
print("DW:", SQLITE_PATH)
print("Parquet:", f".\\{PARQUET_PATH}")
print("Range usado:", d1, "->", d2)
print(f"Tamanho: DW≈{sqlite_size_mb(SQLITE_PATH_ABS, conn):,.1f} MB | Parquet≈{parquet_size_mb(PARQUET_PATH_ABS):,.1f} MB")

# ==========================================================
# 2) DESCOBRE COLUNAS DO PARQUET (case-insensitive)
# ==========================================================
import pyarrow.parquet as pq
pf = pq.ParquetFile(str(PARQUET_PATH))
parquet_cols = pf.schema_arrow.names

p_col_data    = find_col_case_insensitive(parquet_cols, "data")
p_col_hora    = find_col_case_insensitive(parquet_cols, "hora")
p_col_planta  = find_col_case_insensitive(parquet_cols, "planta")
p_col_cadastro= find_col_case_insensitive(parquet_cols, "cadastro")

if not p_col_data or not p_col_cadastro:
    raise ValueError(f"Não achei colunas mínimas no Parquet. Preciso de 'data' e 'cadastro'. Colunas: {parquet_cols}")

# ==========================================================
# 3) BENCHMARK: PARQUET vs DW (mesmas métricas)
# ==========================================================
results = []

# ---------------------------
# (A) Presença diária única
#   Parquet: count de grupos (cadastro,data,planta)
#   DW: agg_diarios_unico count (range)
# ---------------------------
if HAS_DUCKDB:
    dcon = duckdb.connect(database=":memory:")
    # Parquet
    t0 = time.perf_counter()
    q = f"""
        SELECT COUNT(*) AS n
        FROM (
            SELECT 1
            FROM read_parquet('{PARQUET_PATH.as_posix()}')
            WHERE CAST({p_col_data} AS DATE) BETWEEN DATE '{d1}' AND DATE '{d2}'
            GROUP BY {p_col_cadastro}, CAST({p_col_data} AS DATE), {p_col_planta if p_col_planta else "NULL"}
        )
    """
    parq_n = dcon.execute(q).fetchone()[0]
    parq_ms = (time.perf_counter() - t0) * 1000
else:
    # pandas fallback: lê só colunas necessárias e filtra range (pode demorar/usar RAM)
    use_cols = [p_col_cadastro, p_col_data] + ([p_col_planta] if p_col_planta else [])
    t0 = time.perf_counter()
    dfp = pd.read_parquet(PARQUET_PATH, columns=use_cols)
    dfp[p_col_data] = pd.to_datetime(dfp[p_col_data]).dt.date
    d1d, d2d = pd.to_datetime(d1).date(), pd.to_datetime(d2).date()
    dfp = dfp[(dfp[p_col_data] >= d1d) & (dfp[p_col_data] <= d2d)]
    grp_cols = [p_col_cadastro, p_col_data] + ([p_col_planta] if p_col_planta else [])
    parq_n = dfp.groupby(grp_cols, dropna=False).size().shape[0]
    parq_ms = (time.perf_counter() - t0) * 1000

dw_ms, dw_row = one(conn, """
    SELECT COUNT(*) FROM agg_diarios_unico
    WHERE data BETWEEN ? AND ?
""", (d1, d2))
dw_n = int(dw_row[0])
results.append(("presenca_diaria_unica", "parquet", float(parq_ms), int(parq_n)))
results.append(("presenca_diaria_unica", "dw_agg_diarios_unico", float(dw_ms), int(dw_n)))

# ---------------------------
# (B) Registros semanais
#   Parquet: count de grupos (cadastro, ano, semana)
#   DW: agg_registros_semanais count (filtrando por data se existir)
# ---------------------------
if HAS_DUCKDB:
    t0 = time.perf_counter()
    q = f"""
        SELECT COUNT(*) AS n
        FROM (
            SELECT 1
            FROM read_parquet('{PARQUET_PATH.as_posix()}')
            WHERE CAST({p_col_data} AS DATE) BETWEEN DATE '{d1}' AND DATE '{d2}'
            GROUP BY
              {p_col_cadastro},
              EXTRACT(year FROM CAST({p_col_data} AS DATE)),
              EXTRACT(week FROM CAST({p_col_data} AS DATE))
        )
    """
    parq_n = dcon.execute(q).fetchone()[0]
    parq_ms = (time.perf_counter() - t0) * 1000
else:
    use_cols = [p_col_cadastro, p_col_data]
    t0 = time.perf_counter()
    dfp = pd.read_parquet(PARQUET_PATH, columns=use_cols)
    dt = pd.to_datetime(dfp[p_col_data])
    dfp["_date"] = dt.dt.date
    d1d, d2d = pd.to_datetime(d1).date(), pd.to_datetime(d2).date()
    dfp = dfp[(dfp["_date"] >= d1d) & (dfp["_date"] <= d2d)]
    dfp["_ano"] = dt.dt.year
    # semana ISO (mais estável para comparar)
    dfp["_semana"] = dt.dt.isocalendar().week.astype(int)
    parq_n = dfp.groupby([p_col_cadastro, "_ano", "_semana"], dropna=False).size().shape[0]
    parq_ms = (time.perf_counter() - t0) * 1000

if exists_sqlite(conn, "agg_registros_semanais", "table"):
    # tenta filtrar por data se a coluna existir
    cols = [c[1] for c in conn.execute("PRAGMA table_info('agg_registros_semanais')").fetchall()]
    if "data" in cols:
        dw_ms, dw_row = one(conn, """
            SELECT COUNT(*) FROM agg_registros_semanais
            WHERE data BETWEEN ? AND ?
        """, (d1, d2))
    else:
        dw_ms, dw_row = one(conn, "SELECT COUNT(*) FROM agg_registros_semanais")
    dw_n = int(dw_row[0])
else:
    dw_ms, dw_n = None, None

results.append(("registros_semanais", "parquet", float(parq_ms), int(parq_n)))
if dw_ms is not None:
    results.append(("registros_semanais", "dw_agg_registros_semanais", float(dw_ms), int(dw_n)))

# ---------------------------
# (C) Semana enriquecida (23–25)
#   Parquet: count de grupos (cadastro, ano, semana) com filtro de ano
#   DW: agg_semana_enriquecida count (ano IN 2023..2025)
# ---------------------------
if HAS_DUCKDB:
    t0 = time.perf_counter()
    q = f"""
        SELECT COUNT(*) AS n
        FROM (
            SELECT 1
            FROM read_parquet('{PARQUET_PATH.as_posix()}')
            WHERE EXTRACT(year FROM CAST({p_col_data} AS DATE)) IN (2023, 2024, 2025)
            GROUP BY
              {p_col_cadastro},
              EXTRACT(year FROM CAST({p_col_data} AS DATE)),
              EXTRACT(week FROM CAST({p_col_data} AS DATE))
        )
    """
    parq_n = dcon.execute(q).fetchone()[0]
    parq_ms = (time.perf_counter() - t0) * 1000
else:
    use_cols = [p_col_cadastro, p_col_data]
    t0 = time.perf_counter()
    dfp = pd.read_parquet(PARQUET_PATH, columns=use_cols)
    dt = pd.to_datetime(dfp[p_col_data])
    dfp["_ano"] = dt.dt.year
    dfp = dfp[dfp["_ano"].isin([2023, 2024, 2025])]
    dfp["_semana"] = dt.dt.isocalendar().week.astype(int)
    parq_n = dfp.groupby([p_col_cadastro, "_ano", "_semana"], dropna=False).size().shape[0]
    parq_ms = (time.perf_counter() - t0) * 1000

dw_ms, dw_row = one(conn, """
    SELECT COUNT(*) FROM agg_semana_enriquecida
    WHERE ano IN (2023, 2024, 2025)
""")
dw_n = int(dw_row[0])
results.append(("semana_enriquecida_23_25", "parquet", float(parq_ms), int(parq_n)))
results.append(("semana_enriquecida_23_25", "dw_agg_semana_enriquecida", float(dw_ms), int(dw_n)))

# ---------------------------
# (D) Almoço (11–14)
#   Parquet: filtro hora 11:00:00–14:00:00 (se existir hora)
#   DW: agg_almoco filtro hora
# ---------------------------
if p_col_hora:
    if HAS_DUCKDB:
        t0 = time.perf_counter()
        q = f"""
            SELECT COUNT(*) AS n
            FROM read_parquet('{PARQUET_PATH.as_posix()}')
            WHERE CAST({p_col_data} AS DATE) BETWEEN DATE '{d1}' AND DATE '{d2}'
              AND CAST({p_col_hora} AS TIME) BETWEEN TIME '11:00:00' AND TIME '14:00:00'
        """
        parq_n = dcon.execute(q).fetchone()[0]
        parq_ms = (time.perf_counter() - t0) * 1000
    else:
        use_cols = [p_col_data, p_col_hora]
        t0 = time.perf_counter()
        dfp = pd.read_parquet(PARQUET_PATH, columns=use_cols)
        dt = pd.to_datetime(dfp[p_col_data])
        dfp["_date"] = dt.dt.date
        d1d, d2d = pd.to_datetime(d1).date(), pd.to_datetime(d2).date()
        dfp = dfp[(dfp["_date"] >= d1d) & (dfp["_date"] <= d2d)]
        # compara strings HH:MM:SS (funciona se vier nesse formato)
        h = dfp[p_col_hora].astype(str)
        parq_n = int(((h >= "11:00:00") & (h <= "14:00:00")).sum())
        parq_ms = (time.perf_counter() - t0) * 1000

    results.append(("almoco_11_14", "parquet", float(parq_ms), int(parq_n)))

if exists_sqlite(conn, "agg_almoco", "table"):
    dw_ms, dw_row = one(conn, """
        SELECT COUNT(*) FROM agg_almoco
        WHERE hora BETWEEN '11:00:00' AND '14:00:00'
    """)
    results.append(("almoco_11_14", "dw_agg_almoco", float(dw_ms), int(dw_row[0])))

# ==========================================================
# 4) RESULTADOS + SPEEDUP
# ==========================================================
df = pd.DataFrame(results, columns=["metrica", "fonte", "ms", "linhas"])
display(df)

# speedup: parquet_ms / dw_ms (x) e melhora%
speed = []
for m in df["metrica"].unique():
    p = df[(df["metrica"] == m) & (df["fonte"] == "parquet")]
    d = df[(df["metrica"] == m) & (df["fonte"].str.startswith("dw_"))]
    if p.empty or d.empty:
        continue
    p_ms = float(p.iloc[0]["ms"])
    d_ms = float(d.sort_values("ms").iloc[0]["ms"])
    d_src = d.sort_values("ms").iloc[0]["fonte"]
    speed.append({
        "metrica": m,
        "parquet_ms": p_ms,
        "dw_ms": d_ms,
        "dw_fonte": d_src,
        "speedup_x": (p_ms / d_ms) if d_ms > 0 else None,
        "melhora_pct": (1 - (d_ms / p_ms)) * 100 if p_ms > 0 else None,
    })
df_speed = pd.DataFrame(speed).sort_values("speedup_x", ascending=False)
display(df_speed)

# ==========================================================
# 5) GRÁFICOS
# ==========================================================
# (1) Tempo (ms) - Parquet vs DW por métrica
piv = df.pivot_table(index="metrica", columns="fonte", values="ms", aggfunc="min")
piv = piv.loc[[m for m in df["metrica"].unique() if m in piv.index]]

plt.figure(figsize=(12,4))
x = range(len(piv.index))
cols = list(piv.columns)
width = 0.8 / max(1, len(cols))

for i, col in enumerate(cols):
    plt.bar([xi + i*width for xi in x], piv[col].values, width=width, label=col)

plt.xticks([xi + width*(len(cols)-1)/2 for xi in x], piv.index, rotation=0)
plt.yscale("log")
plt.ylabel("Tempo (ms) - escala log")
plt.title(f"Parquet vs DW: tempo por métrica (range {d1} -> {d2})")
plt.legend()
plt.tight_layout()
plt.show()

# (2) Impacto em CÉLULAS (rows×cols): Parquet vs DW (serving-only) vs DW com RAW (estimado)
# Ideia: mostrar quanto a remoção da tabela gigante 'evento_acesso_flat' reduz o DW em termos de "células".

def sqlite_tables(conn):
    return [r[0] for r in conn.execute("""
        SELECT name FROM sqlite_master
        WHERE type='table' AND name NOT LIKE 'sqlite_%'
    """).fetchall()]

def sqlite_cols(conn, table):
    return [c[1] for c in conn.execute(f"PRAGMA table_info('{table}')").fetchall()]

def sqlite_rows_est(conn, table):
    # tenta sqlite_stat1 (rápido); fallback COUNT(*)
    try:
        r = conn.execute("SELECT stat FROM sqlite_stat1 WHERE tbl=? LIMIT 1", (table,)).fetchone()
        if r and r[0]:
            return int(r[0].split()[0])
    except Exception:
        pass
    return int(conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])

# Células do Parquet (meta, sem carregar)
parquet_rows = pf.metadata.num_rows
parquet_cols_n = getattr(pf.schema_arrow, 'num_fields', len(pf.schema_arrow))
parquet_cells = int(parquet_rows) * int(parquet_cols_n)

# Células do DW atual (somatório por tabela)
dw_tables = sqlite_tables(conn)
dw_cells_current = 0
dw_cells_breakdown = []
for t in dw_tables:
    cols_n = len(sqlite_cols(conn, t))
    rows_n = sqlite_rows_est(conn, t)
    cells_n = int(rows_n) * int(cols_n)
    dw_cells_current += cells_n
    dw_cells_breakdown.append((t, rows_n, cols_n, cells_n))

# Estimativa do DW "antes" (com RAW) caso a tabela não exista (serving-only)
BASELINE_EVENTO_ACESSO_FLAT_ROWS = 35_490_730  # valor obtido no seu relatório anterior
BASELINE_EVENTO_ACESSO_FLAT_COLS = 12          # cols_dw do evento_acesso_flat

if 'evento_acesso_flat' in dw_tables:
    flat_rows = sqlite_rows_est(conn, 'evento_acesso_flat')
    flat_cols = len(sqlite_cols(conn, 'evento_acesso_flat'))
    dw_cells_with_raw = dw_cells_current
else:
    flat_rows = BASELINE_EVENTO_ACESSO_FLAT_ROWS
    flat_cols = BASELINE_EVENTO_ACESSO_FLAT_COLS
    dw_cells_with_raw = dw_cells_current + (flat_rows * flat_cols)

# Prints rápidos (para ficar explícito)
print(f"Parquet: rows={parquet_rows:,} cols={parquet_cols_n:,} cells={parquet_cells:,}")
print(f"DW atual: tables={len(dw_tables)} cells={dw_cells_current:,}")
print(f"DW c/ RAW (estimado): +evento_acesso_flat({flat_rows:,}×{flat_cols}) => cells={dw_cells_with_raw:,}")
print(f"DW atual / Parquet: {dw_cells_current / parquet_cells:.4f}x")
print(f"DW c/ RAW / Parquet: {dw_cells_with_raw / parquet_cells:.4f}x")
print(f"Redução (cells) ao remover RAW: {(1 - (dw_cells_current / dw_cells_with_raw)) * 100:.1f}%" )

# Gráfico 2: comparação de células (log) + destaque do impacto
labels2 = ["Parquet(df_ronda)", "DW c/ RAW (estimado)", "DW atual (serving-only)"]
vals2   = [parquet_cells, dw_cells_with_raw, dw_cells_current]

plt.figure(figsize=(11,4))
plt.bar(labels2, vals2)
plt.yscale('log')
plt.ylabel('Células (rows×cols) - escala log')
plt.title('Impacto da otimização: redução de células no DW vs df_ronda')
plt.tight_layout()
plt.show()
# (3) Comparativo de tamanho: Parquet vs DW
dw_mb = sqlite_size_mb(SQLITE_PATH, conn)
pq_mb = parquet_size_mb(PARQUET_PATH)

plt.figure(figsize=(6,4))
plt.bar(["Parquet", "DW (SQLite)"], [pq_mb, dw_mb])
plt.ylabel("Tamanho (MB)")
plt.title("Tamanho em disco: Parquet vs DW")
plt.tight_layout()
plt.show()

conn.close()
if HAS_DUCKDB:
    dcon.close()